# Simple clustering


In [ ]:
# Resolve shared multiclust imports when this notebook is run from notebooks/<project>/.
from pathlib import Path
import os
import sys

_candidates = []
if os.environ.get("MULTICLUST_ROOT"):
    _candidates.append(Path(os.environ["MULTICLUST_ROOT"]).expanduser())
_candidates.extend([Path.cwd(), *Path.cwd().parents])
for _candidate in _candidates:
    if (_candidate / "Utils.py").exists() and (_candidate / "full_pipeline.py").exists():
        MULTICLUST_ROOT = _candidate.resolve()
        break
else:
    raise RuntimeError("Could not find multiclust root. Set MULTICLUST_ROOT to the Code/multiclust folder.")
if str(MULTICLUST_ROOT) not in sys.path:
    sys.path.insert(0, str(MULTICLUST_ROOT))
print(f"Using multiclust root: {MULTICLUST_ROOT}")


In [ ]:
# Import theme for plots
import importlib
import theme
importlib.reload(theme)
theme.apply_all()
from theme import CC_COLOR


# Data Preparation

### Load all requirements

In [ ]:
import pandas as pd
pd.__version__


In [ ]:
from Utils import *
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import pickle
import dill
from SVM import *

#Import own functions
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list
import scipy.cluster.hierarchy as hierarchy

import seaborn as sns
from scipy.stats import f_oneway, kruskal, shapiro, levene, chi2_contingency, fisher_exact

import importlib


### Load in data and metatable

In [ ]:
data = pd.read_csv("path/to/simpleclust_data/merged_data_simpleclust.csv")
meta = pd.read_csv("path/to/simpleclust_data/merged_meta_simpleclust.csv")



Change colums to character type when categorical.

In [ ]:

# Load the dictionary from an Excel file
dictionary_DIR = "path/to/data_dictionary.xlsx"  
# Read the dictionary Excel file
dict_df = pd.read_excel(dictionary_DIR)
missing_dictionary_codes = {-900, -300, -997, -998, -999}

# Infer datatype for each note using a vectorized approach with apply and ifelse-like logic.
dict_df["InferredDataType"] = dict_df["Notes"].apply(lambda note: 
    (
        "Categorical" 
        if len(set(float(k) for k in re.findall(r"(-?\d+)\s*=\s*[^,;]+", note) if float(k) not in missing_dictionary_codes)) >= 2 
        else np.nan
    ) if pd.notna(note) and note != "" else np.nan
)

# Retain this list for downstream reporting. Do not cast these columns to
# string: numeric ordinal scores are already correctly ordered numerically,
# and dummy_code() now protects numeric-like object columns from one-hot encoding.
cat_vars = dict_df.loc[dict_df["InferredDataType"] == "Categorical", "ElementName"].tolist()
data.replace(["nan", "NaN", "None", "NULL", "null", ""], np.nan, inplace=True)


### Remove community controls

In [ ]:
remove_CC = True

if remove_CC:
    # Split CHR and community controls (HC) so CC can be preprocessed
    # with the fitted clustering pipeline later on.
    data_CHR = data[data['phenotype'] != 'HC'].reset_index(drop=True)
    data_CC = data[data['phenotype'] == 'HC'].reset_index(drop=True)



### Split into discovery and test

In [ ]:
# 1. Load the Prescient ID list once:
prescient = pd.read_csv(
    "path/to/"
    "project/"
    "Data/Prescient_Client List_ DPACC ID with RA Name (ALL sites)_20_03_2025.txt",
    sep="	"
)
prescient.columns = ["fkLocationID", "pkclientid", "src_subject_id", "RAname"]
prescient_ids = set(prescient["src_subject_id"])

# 2. Split baseline CHR and CC into discovery/test
#    so the same cohort split is preserved throughout the notebook.
discovery_data, test_data = split_by_network(data_CHR, prescient_ids)
discovery_data_CC, test_data_CC = split_by_network(data_CC, prescient_ids)



In [ ]:
test_data_CC

### Data cleaning

Handle missing values. 
First remove columns with more than 50% missing and then removing rows that have more than 50% missing. #TODO! Check later how much % you actually want to use. 
Right now we're using a simple model to impute the data Important to look at this and choose an appropriate way to deal with missing data. TODO!

In [ ]:
cleaned_discovery, cleaned_test = remove_high_missing_data_split(
    discovery_data, test_data,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)

display(cleaned_discovery)

cleaned_discovery_CC, cleaned_test_CC = remove_high_missing_data_split(
    discovery_data_CC, test_data_CC,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)



In [ ]:
# Only keep relevant modalities
modalities_keep = ["Include_cluster"]

vars_to_keep = meta["ElementName"][meta["Modality"].isin(modalities_keep)].tolist()
vars_to_keep.append("src_subject_id")
vars_to_keep.append("phenotype")

cleaned_discovery = cleaned_discovery[cleaned_discovery.columns.intersection(vars_to_keep)]
cleaned_discovery_CC = cleaned_discovery_CC[cleaned_discovery_CC.columns.intersection(vars_to_keep)]



In [ ]:
cleaned_discovery


In [ ]:
# Save the cleaned dataframes to a CSV file
cleaned_discovery.to_csv(
    "path/to/"
    "simpleclust_data/cleaned_discovery_data_simpleclust.csv",
    index=False
)

cleaned_test.to_csv(
    "path/to/simpleclust_data/cleaned_test_data_simpleclust.csv",
    index=False
)


### Collinearity diagnostic

This diagnostic is restricted to the single Simpleclust domain, `Include_cluster`. VIF and the tabular diagnostics use the pipeline-preprocessed matrix, while the Pearson and Spearman heatmaps use the unique raw clustering variables so dummy-coded levels are not displayed as duplicate variables. It does not automatically remove variables.

In [ ]:
simpleclust_domain = "Include_cluster"

_, collinearity_subject_ids, collinearity_processed_modalities = preprocessing(
    cleaned_discovery,
    meta,
    subject_id_column="src_subject_id",
    col_threshold=0.5,
    row_threshold=0.5,
    skew_threshold=0.75,
    scaler_type="robust",
    modalities=[simpleclust_domain],
    dummy_code_modalities=[simpleclust_domain],
    mixed_categorical_modalities=[],
)

if set(collinearity_processed_modalities) != {simpleclust_domain}:
    raise RuntimeError(
        "Simpleclust collinearity must run only for Include_cluster; "
        f"received {sorted(collinearity_processed_modalities)}"
    )

simpleclust_collinearity = assess_domain_collinearity(
    {simpleclust_domain: collinearity_processed_modalities[simpleclust_domain]},
    correlation_methods=("pearson", "spearman"),
    high_corr_threshold=0.90,
    moderate_corr_threshold=0.70,
)

collinearity_summary = simpleclust_collinearity["summary"]
domain_collinearity_correlation_pairs = simpleclust_collinearity["correlation_pairs"]
domain_collinearity_vif = simpleclust_collinearity["vif"]
domain_collinearity_condition_index = simpleclust_collinearity["condition_index"]
domain_collinearity_skipped_variables = simpleclust_collinearity["skipped_variables"]

collinearity_output_dir = Path(
    "path/to/"
    "simpleclust_data"
)
collinearity_outputs = {
    "summary": collinearity_output_dir / "simpleclust_collinearity_summary.csv",
    "correlation_pairs": collinearity_output_dir / "simpleclust_collinearity_correlation_pairs.csv",
    "vif": collinearity_output_dir / "simpleclust_collinearity_vif_tolerance.csv",
    "condition_index": collinearity_output_dir / "simpleclust_collinearity_condition_index.csv",
    "skipped_variables": collinearity_output_dir / "simpleclust_collinearity_skipped_variables.csv",
}
for result_name, output_path in collinearity_outputs.items():
    simpleclust_collinearity[result_name].to_csv(output_path, index=False)
    print(f"Saved {result_name}: {output_path}")

simpleclust_raw_variables = (
    meta.loc[meta["Modality"].eq(simpleclust_domain), "ElementName"]
    .dropna()
    .drop_duplicates()
    .loc[lambda variables: variables.isin(cleaned_discovery.columns)]
    .tolist()
)
simpleclust_raw_variables = order_simpleclust_features(simpleclust_raw_variables)
simpleclust_heatmap_data = cleaned_discovery[
    ["src_subject_id"] + simpleclust_raw_variables
].copy()
if simpleclust_heatmap_data.columns.duplicated().any():
    duplicate_columns = simpleclust_heatmap_data.columns[
        simpleclust_heatmap_data.columns.duplicated()
    ].tolist()
    raise RuntimeError(f"Duplicate raw heatmap columns: {duplicate_columns}")
print(
    f"Collinearity heatmaps use {len(simpleclust_raw_variables)} unique raw "
    f"{simpleclust_domain} variables."
)

collinearity_heatmap_paths = plot_domain_collinearity_heatmaps(
    {simpleclust_domain: simpleclust_heatmap_data},
    output_dir=collinearity_output_dir,
    subject_id_column="src_subject_id",
    correlation_methods=("pearson", "spearman"),
    file_prefix="simpleclust_raw_variable_collinearity",
)
for heatmap_path in collinearity_heatmap_paths:
    print(f"Saved heatmap: {heatmap_path}")

print("Domain-level collinearity summary")
display(collinearity_summary)
print("Pairs with |r| >= 0.70")
display(domain_collinearity_correlation_pairs)
print("VIF and tolerance")
display(domain_collinearity_vif)
print("Condition index")
display(domain_collinearity_condition_index)


# Full pipeline results

## Load in results

The actual clustering pipeline is run on Spartan using the full pipeline scripts.

In [ ]:

from pathlib import Path
import glob
import os

from matplotlib.lines import Line2D
from pandas.plotting import scatter_matrix
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.manifold import TSNE

DIMRED_ROOT = Path('path/to/results/Simpleclust/release/Simpleclust_100bcs_dimruns')


def _extract_fold_number(fold_name):
    digits = ''.join(ch for ch in str(fold_name) if ch.isdigit())
    return int(digits) if digits else fold_name


def _mode_value(series):
    series = series.dropna()
    if series.empty:
        return None
    if series.apply(lambda x: isinstance(x, (list, tuple))).any():
        expanded = pd.DataFrame(series.tolist())
        return expanded.mode(dropna=True).iloc[0].tolist()
    mode = series.mode(dropna=True)
    return mode.iloc[0] if not mode.empty else series.iloc[0]


def _safe_numeric(value):
    if isinstance(value, (list, tuple, dict, pd.Series, pd.DataFrame, np.ndarray)):
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan


def summarise_feature_differences(final_metrics, top_k=10):
    data = final_metrics.get('data')
    labels = np.asarray(final_metrics.get('final_labels'))
    if data is None or 'src_subject_id' not in data.columns:
        return pd.DataFrame()

    feature_df = data.drop(columns=['src_subject_id'], errors='ignore').select_dtypes(include=[np.number]).copy()
    if feature_df.empty or len(labels) != len(feature_df) or pd.Series(labels).nunique() < 2:
        return pd.DataFrame()

    valid_cols = feature_df.columns[feature_df.nunique(dropna=False) > 1]
    feature_df = feature_df.loc[:, valid_cols]
    if feature_df.empty:
        return pd.DataFrame()

    f_vals, p_vals = f_classif(feature_df.fillna(feature_df.mean()), labels)
    out = pd.DataFrame({
        'feature': feature_df.columns,
        'f_value': f_vals,
        'p_value': p_vals,
    })
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=['f_value'])
    return out.sort_values(['f_value', 'p_value'], ascending=[False, True]).head(top_k).reset_index(drop=True)


def plot_autoencoder_diagnostics(run_name, final_metrics):
    ae_res = final_metrics.get('ae_res')
    if not isinstance(ae_res, dict):
        print(f'{run_name}: no ae_res payload found, skipping latent diagnostics.')
        return
    required = {'final_latent', 'all_true', 'all_pred'}
    if not required.issubset(ae_res):
        print(f'{run_name}: ae_res missing {sorted(required - set(ae_res))}, skipping latent diagnostics.')
        return

    test_latent = ae_res['final_latent']
    X_true = np.asarray(ae_res['all_true']).ravel()
    X_pred = np.asarray(ae_res['all_pred']).ravel()

    plt.figure(figsize=(6, 6))
    plt.scatter(X_true, X_pred, alpha=0.35)
    plt.xlabel('Original Data')
    plt.ylabel('Reconstructed Data')
    plt.title(f'{run_name}: Original vs Reconstructed Data')
    min_val = min(np.nanmin(X_true), np.nanmin(X_pred))
    max_val = max(np.nanmax(X_true), np.nanmax(X_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    plt.tight_layout()
    plt.show()

    latent_arr = np.asarray(test_latent)
    if latent_arr.ndim == 1:
        latent_arr = latent_arr.reshape(-1, 1)

    plt.figure(figsize=(6, 4))
    plt.hist(latent_arr.ravel(), bins=np.linspace(-2, 2, 50))
    plt.title(f'{run_name}: Latent Distribution')
    plt.tight_layout()
    plt.show()

    if isinstance(test_latent, pd.DataFrame):
        latent_df = test_latent.copy()
    else:
        latent_df = pd.DataFrame(latent_arr, columns=[f'z{i}' for i in range(latent_arr.shape[1])])

    scatter_matrix(latent_df, alpha=0.5, figsize=(8, 8), diagonal='hist')
    plt.suptitle(f'{run_name}: Pairwise Latent Scatter Plots')
    plt.tight_layout()
    plt.show()


dimred_runs = {}
fold_summary_rows = []
final_summary_rows = []
feature_difference_tables = {}

for run_dir in sorted(DIMRED_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue

    run_name = run_dir.name
    results_dir = run_dir / 'results'
    plots_dir = run_dir / 'plots'
    plots_dir.mkdir(exist_ok=True)

    metrics_files = sorted(results_dir.glob('fold*/metrics.pkl'))
    final_metric_files = sorted(results_dir.glob('final/final_metrics.pkl'))
    if not metrics_files or not final_metric_files:
        print(f'Skipping {run_name}: missing fold or final metric files.')
        continue

    metrics = {}
    for metrics_file in metrics_files:
        fold_name = metrics_file.parent.name
        with open(metrics_file, 'rb') as f:
            metrics[fold_name] = pickle.load(f)

    with open(final_metric_files[0], 'rb') as f:
        final_metrics_obj = pickle.load(f)

    dimred_runs[run_name] = {
        'base': str(run_dir),
        'results_dir': str(results_dir),
        'plots_dir': str(plots_dir),
        'metrics': metrics,
        'final_metrics': final_metrics_obj,
    }

    for fold_name, fold_data in metrics.items():
        best_fitness = fold_data.get('best_fitness', {})
        metrics_summary = best_fitness.get('metrics_summary', {})
        stab_sum_mat = metrics_summary.get('stab_SUM_MAT', {})
        fold_summary_rows.append({
            'dimred_option': run_name,
            'fold': fold_name,
            'fold_order': _extract_fold_number(fold_name),
            'fold_status': fold_data.get('fold_status', 'completed'),
            'skip_reason': fold_data.get('skip_reason'),
            'best_params': fold_data.get('best_params'),
            'best_stability': _safe_numeric(best_fitness.get('stability', metrics_summary.get('stab_ari', fold_data.get('stability')))),
            'best_quality': _safe_numeric(best_fitness.get('quality', metrics_summary.get('quality', fold_data.get('quality')))),
            'best_ari': _safe_numeric(metrics_summary.get('stab_ari', best_fitness.get('stability'))),
            'best_ccc': _safe_numeric(stab_sum_mat.get('CCC', fold_data.get('CCC'))),
        })

    run_fold_df = pd.DataFrame([row for row in fold_summary_rows if row['dimred_option'] == run_name]).sort_values('fold_order')
    params_mode = None
    skipped_folds = []
    folds_with_params = 0
    if not run_fold_df.empty:
        has_params = run_fold_df['best_params'].apply(lambda x: isinstance(x, dict))
        folds_with_params = int(has_params.sum())
        skipped_folds = run_fold_df.loc[~has_params, 'fold'].tolist()
        valid_params = run_fold_df.loc[has_params, 'best_params']
        if not valid_params.empty:
            params_df = pd.DataFrame(valid_params.tolist())
            params_mode = {col: _mode_value(params_df[col]) for col in params_df.columns}

    stab_sum_final = final_metrics_obj.get('final_stability_SUM_MAT_full', {})
    final_summary_rows.append({
        'dimred_option': run_name,
        'folds_loaded': len(metrics),
        'folds_with_params': folds_with_params,
        'skipped_folds': len(skipped_folds),
        'fold_param_mode': params_mode,
        'final_params': final_metrics_obj.get('final_params'),
        'final_quality': _safe_numeric(final_metrics_obj.get('final_quality')),
        'final_stability': _safe_numeric(final_metrics_obj.get('final_stability')),
        'final_stability_ari': _safe_numeric(final_metrics_obj.get('final_stability_ari', final_metrics_obj.get('final_stability'))),
        'final_ccc': _safe_numeric(stab_sum_final.get('CCC')),
        'final_pac': _safe_numeric(stab_sum_final.get('PAC')),
        'final_quality_pvalue': _safe_numeric(final_metrics_obj.get('final_quality_pvalue')),
        'final_stability_ari_pvalue': _safe_numeric(final_metrics_obj.get('final_stability_ari_pvalue')),
    })

    feature_difference_tables[run_name] = summarise_feature_differences(final_metrics_obj, top_k=10)


dimred_fold_summary_df = pd.DataFrame(fold_summary_rows).sort_values(['dimred_option', 'fold_order']).reset_index(drop=True)
dimred_run_overview_df = pd.DataFrame(final_summary_rows)

if dimred_run_overview_df.empty:
    raise RuntimeError(f'No complete dimensionality-reduction runs found in {DIMRED_ROOT}.')

dimred_run_overview_df['stability_rank'] = dimred_run_overview_df['final_stability_ari'].rank(
    ascending=False,
    method='min'
)
dimred_run_overview_df['quality_rank'] = dimred_run_overview_df['final_quality'].rank(
    ascending=False,
    method='min'
)

n_runs = len(dimred_run_overview_df)
dimred_run_overview_df['stability_score'] = (n_runs - dimred_run_overview_df['stability_rank'] + 1) / n_runs
dimred_run_overview_df['quality_score'] = (n_runs - dimred_run_overview_df['quality_rank'] + 1) / n_runs

# Keep all runs in the overview table, but only allow complete 5-fold runs
# to be selected for downstream analyses.
required_complete_folds = 5
dimred_run_overview_df['eligible_for_selection'] = (
    (dimred_run_overview_df['folds_loaded'] == required_complete_folds)
    & (dimred_run_overview_df['folds_with_params'] == required_complete_folds)
    & (dimred_run_overview_df['skipped_folds'] == 0)
)

# Select the downstream run using both stability and quality, with stability weighted more heavily.
dimred_run_overview_df['selection_score'] = (
    0.5 * dimred_run_overview_df['stability_score']
    + 0.5 * dimred_run_overview_df['quality_score']
)

dimred_run_overview_df = dimred_run_overview_df.sort_values(
    ['selection_score', 'final_stability_ari', 'final_quality', 'final_ccc'],
    ascending=[False, False, False, False]
).reset_index(drop=True)

eligible_dimred_overview_df = dimred_run_overview_df.loc[
    dimred_run_overview_df['eligible_for_selection']
].copy()
if eligible_dimred_overview_df.empty:
    raise RuntimeError(
        'No dimensionality-reduction run had complete valid fold support '
        f'({required_complete_folds}/{required_complete_folds} folds).'
    )

best_dimred_option = eligible_dimred_overview_df.iloc[0]['dimred_option']
selected_dimred = dimred_runs[best_dimred_option]
base = selected_dimred['base']
results_dir = selected_dimred['results_dir']
plots_dir = selected_dimred['plots_dir']
metrics = selected_dimred['metrics']
final_metrics = selected_dimred['final_metrics']

print(f'Loaded {len(dimred_runs)} dimensionality-reduction runs from: {DIMRED_ROOT}')
print(f'Selected best run for downstream analyses: {best_dimred_option}')
print('Selection rule: rank only complete 5-fold runs, using 50% stability rank + 50% quality rank, then tie-break on stability, quality, and CCC.')
display(dimred_run_overview_df)


In [ ]:

print('Best parameters in folds across dimensionality-reduction options:')
display(dimred_fold_summary_df[['dimred_option', 'fold', 'best_params']])


In [ ]:

print('Most common best parameters across folds for each dimensionality-reduction option:')
display(dimred_run_overview_df[['dimred_option', 'fold_param_mode']])


## Check quality of clusters

### Stability of best individual in each fold

In [ ]:

print('Quality of clusters across folds:')
display(
    dimred_fold_summary_df[
        ['dimred_option', 'fold', 'best_quality', 'best_stability', 'best_ari', 'best_ccc']
    ]
)


# Final on all data results

### Load in resutls and check quality

In [ ]:

print(f'Loaded best run for downstream notebook analyses from: {base}')
print(f'Available dimensionality-reduction options: {sorted(dimred_runs)}')


In [ ]:
print('Final parameters across dimensionality-reduction options:')
display(dimred_run_overview_df[['dimred_option', 'final_params']])
print(f'\nDownstream analyses will use: {best_dimred_option}')
print('Final parameters used in the downstream model:', final_metrics['final_params'])


In [ ]:
final_metrics.keys()


In [ ]:
final_metrics['data'].columns

In [ ]:
print('Final quality, ARI, and CCC across dimensionality-reduction options:')
display(
    dimred_run_overview_df[
        ['dimred_option', 'final_quality', 'final_stability_ari', 'final_ccc', 'final_pac']
    ]
)

print('\nSelected best run summary:')
print('Composite quality score:', final_metrics['final_quality'])
print('Stability (with ARI):', final_metrics.get('final_stability_ari', final_metrics['final_stability']))


In [ ]:
final_metrics.keys()


In [ ]:

print('P-values across dimensionality-reduction options:')
display(
    dimred_run_overview_df[
        ['dimred_option', 'final_quality_pvalue', 'final_stability_ari_pvalue']
    ]
)

print('Final integrated cluster stability (CCC):', final_metrics['final_stability_SUM_MAT_full']['CCC'])
print('Final integrated cluster stability (PAC):', final_metrics['final_stability_SUM_MAT_full']['PAC'])


## VAE results

In [ ]:
print('Top feature differences across dimensionality-reduction options:')
for run_name, feature_df in feature_difference_tables.items():
    print(f'\n=== {run_name} ===')
    if feature_df.empty:
        print('No feature-difference table available.')
    else:
        display(feature_df)


In [ ]:
for run_name in ('ae', 'vae'):
    if run_name not in dimred_runs:
        print(f'{run_name}: run not found in dimred_runs, skipping latent diagnostics.')
        continue
    print(f'\n=== {run_name}: VAE/AE reconstruction diagnostics ===')
    plot_autoencoder_diagnostics(run_name, dimred_runs[run_name]['final_metrics'])


## Cluster validation sensitivity analyses

This section reports the optional no-cluster / continuum sensitivity analyses when they were enabled for the run profile and the exported sensitivity files are present.

In [ ]:
# Optional cluster-validation / continuum sensitivity analyses
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd


def _truthy_profile_value(value):
    return str(value).strip().strip('"\'').upper() in {"TRUE", "1", "YES", "Y"}


def _parse_profile_exports(profile_path):
    exports = {}
    if profile_path is None or not profile_path.exists():
        return exports
    for line in profile_path.read_text().splitlines():
        line = line.strip()
        if not line.startswith("export ") or "=" not in line:
            continue
        key, value = line[len("export "):].split("=", 1)
        value = value.strip()
        # Keep literal defaults such as ${VAR:-0} as-is; simple quoted values are enough for the flag check.
        exports[key.strip()] = value.strip('"\'')
    return exports


def _infer_notebook_profile():
    notebook_dir = Path.cwd().name
    profile_candidates = []
    if notebook_dir == "clinical_paper":
        profile_candidates.append("clinical_paper")
    elif notebook_dir == "multiclust_extended":
        profile_candidates.append("multiclust_extended")
    elif notebook_dir == "prospect":
        profile_candidates.append("prospect")
    elif notebook_dir == "Simpleclust":
        profile_candidates.append("Simpleclust")
    profile_candidates.append(os.environ.get("RUN_PROFILE", ""))
    for candidate in profile_candidates:
        if candidate:
            return candidate
    return None


def _find_repo_root(start):
    start = Path(start).resolve()
    for parent in [start] + list(start.parents):
        if (parent / "run_profiles").is_dir() and (parent / "full_pipeline.py").exists():
            return parent
    return None


def _profile_enabled_for_sensitivity(repo_root, profile_name):
    if not profile_name or repo_root is None:
        return None, None
    profile_path = repo_root / "run_profiles" / f"{profile_name}.sh"
    exports = _parse_profile_exports(profile_path)
    value = exports.get("DO_CLUSTER_VALIDATION_SENSITIVITY", "FALSE")
    return _truthy_profile_value(value), profile_path


def _display_if_available(obj):
    if "display" in globals():
        display(obj)
    else:
        print(obj)


def _get_nested(dct, path, default=np.nan):
    cur = dct
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def _flatten_sensitivity_results(results):
    rows = []
    for solution, payload in results.get("solutions", {}).items():
        rows.append({
            "solution": solution,
            "kind": payload.get("kind"),
            "observed_k": _get_nested(payload, ["observed_quality", "k"]),
            "observed_n": _get_nested(payload, ["observed_quality", "n"]),
            "observed_n_features": _get_nested(payload, ["observed_quality", "n_features"]),
            "observed_composite": _get_nested(payload, ["observed_quality", "composite"]),
            "observed_silhouette": _get_nested(payload, ["observed_quality", "silhouette"]),
            "observed_calinski_harabasz": _get_nested(payload, ["observed_quality", "calinski_harabasz"]),
            "observed_davies_bouldin": _get_nested(payload, ["observed_quality", "davies_bouldin"]),
            "k1_composite": _get_nested(payload, ["uni_cluster_baseline", "quality", "composite"]),
            "pc1_variance_explained": _get_nested(payload, ["pc1_median_split", "pc1_variance_explained"]),
            "pc1_median_composite": _get_nested(payload, ["pc1_median_split", "quality", "composite"]),
            "pc1_median_ari_with_observed": _get_nested(payload, ["pc1_median_split", "ari_with_observed_labels"]),
            "pc1_median_stability_ari": _get_nested(payload, ["pc1_median_split", "bootstrap_stability", "mean_ari"]),
            "pc1_median_stability_sd_ari": _get_nested(payload, ["pc1_median_split", "bootstrap_stability", "sd_ari"]),
            "dip_available": _get_nested(payload, ["dip_test_pc1", "available"], default=False),
            "dip_statistic_pc1": _get_nested(payload, ["dip_test_pc1", "dip"]),
            "dip_p_value_pc1": _get_nested(payload, ["dip_test_pc1", "p_value"]),
            "gap_selected_k_tibshirani": _get_nested(payload, ["gap_statistic", "selected_k_tibshirani_rule"]),
            "gap_selected_k_max_gap": _get_nested(payload, ["gap_statistic", "selected_k_max_gap"]),
            "sigclust_cluster_index": _get_nested(payload, ["sigclust_approx", "observed_cluster_index"]),
            "sigclust_p_value": _get_nested(payload, ["sigclust_approx", "p_value"]),
            "sigclust_null_mean_cluster_index": _get_nested(payload, ["sigclust_approx", "null_mean_cluster_index"]),
            "gaussian_null_p_quality": _get_nested(payload, ["covariance_matched_gaussian_null", "p_quality_ge_observed_recluster"]),
            "gaussian_null_p_stability": _get_nested(payload, ["covariance_matched_gaussian_null", "p_stability_ge_observed_recluster"]),
            "observed_recluster_stability_ari": _get_nested(payload, ["covariance_matched_gaussian_null", "observed_recluster_stability", "mean_ari"]),
            "gaussian_null_mean_stability_ari": _get_nested(payload, ["covariance_matched_gaussian_null", "null_stability_mean_ari"]),
            "gaussian_null_mean_quality": _get_nested(payload, ["covariance_matched_gaussian_null", "null_quality_mean"]),
        })
    return pd.DataFrame(rows)


_repo_root = _find_repo_root(Path.cwd())
_profile_name = "Simpleclust"
_sensitivity_enabled, _profile_path = _profile_enabled_for_sensitivity(_repo_root, _profile_name)

if _sensitivity_enabled is False:
    print(
        "Cluster validation sensitivity analyses were not enabled for "
        f"profile '{_profile_name}' ({_profile_path})."
    )
else:
    _base_results_dir = Path(results_dir) if "results_dir" in globals() else None
    _sensitivity_dir = _base_results_dir / "final" / "cluster_validation_sensitivity" if _base_results_dir else None
    _summary_csv = _sensitivity_dir / "cluster_validation_sensitivity_summary.csv" if _sensitivity_dir else None
    _results_json = _sensitivity_dir / "cluster_validation_sensitivity_results.json" if _sensitivity_dir else None

    if _sensitivity_enabled is None:
        print("Could not infer whether the run profile enabled cluster validation sensitivity analyses.")
    elif _sensitivity_enabled:
        print(f"Cluster validation sensitivity analyses were enabled for profile '{_profile_name}'.")

    if not _summary_csv or not _summary_csv.exists():
        print(
            "No cluster validation sensitivity summary was found. "
            "Expected file:",
            _summary_csv,
        )
    else:
        cluster_validation_sensitivity_summary = pd.read_csv(_summary_csv)
        print("Cluster validation sensitivity summary:", _summary_csv)
        _display_if_available(cluster_validation_sensitivity_summary)

        if _results_json and _results_json.exists():
            with open(_results_json, "r") as f:
                cluster_validation_sensitivity_results = json.load(f)

            print("Included sensitivity tests:")
            for _test in cluster_validation_sensitivity_results.get("included_tests", []):
                print("-", _test)

            cluster_validation_sensitivity_details = _flatten_sensitivity_results(
                cluster_validation_sensitivity_results
            )
            print("Detailed sensitivity metrics:")
            _display_if_available(cluster_validation_sensitivity_details)
        else:
            print("Detailed sensitivity JSON not found:", _results_json)

        _gap_tables = sorted((_sensitivity_dir / "tables").glob("*/*_gap_statistic.csv"))
        cluster_validation_gap_tables = {}
        if _gap_tables:
            print("Gap-statistic tables:")
            for _gap_table in _gap_tables:
                _gap_name = _gap_table.parent.name
                cluster_validation_gap_tables[_gap_name] = pd.read_csv(_gap_table)
                print("-", _gap_table)
                _display_if_available(cluster_validation_gap_tables[_gap_name])

        _plot_files = sorted((_sensitivity_dir / "plots").glob("*.pdf")) + sorted((_sensitivity_dir / "plots").glob("*.png"))
        if _plot_files:
            print("Sensitivity plot files:")
            for _plot_file in _plot_files:
                print("-", _plot_file)

## Matrix plots

#### Matrix plots final labels

In [ ]:
# Grab consensus
M = final_metrics['final_stability_SUM_MAT_full']['consensus']
M = np.asarray(M, dtype=float)

# Safety checks
assert M.ndim == 2 and M.shape[0] == M.shape[1], "Consensus must be square."
M = (M + M.T) / 2.0           # enforce symmetry (just in case)
np.fill_diagonal(M, 1.0)      # conventional

# Distance from consensus
D = 1.0 - M
dvec = squareform(D, checks=False)

# Hierarchical clustering (match your CCC linkage_method if you want)
Z = linkage(dvec, method="average")

# Leaf order and reordered matrix
order = leaves_list(Z)
M_ord = M[np.ix_(order, order)]

# Plot and save into the selected dimensionality-reduction run's plots folder.
plots_dir_path = Path(plots_dir)
plots_dir_path.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(M_ord, aspect='auto', vmin=0, vmax=1)
ax.set_xlabel("Subjects")
ax.set_ylabel("Subjects")
fig.colorbar(im, ax=ax, label="Consensus")
fig.tight_layout()
for ext in ("png", "pdf"):
    output_path = plots_dir_path / f"consensus_matrix_hierarchical_order.{ext}"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print("Saved consensus matrix to:", output_path)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plots_dir_path = Path(plots_dir)
plots_dir_path.mkdir(parents=True, exist_ok=True)

diag = final_metrics['final_stability_SUM_MAT_full']

# --- Consensus matrix ---
M = np.asarray(diag['consensus'], dtype=float)

# Make matrix symmetric and set diagonal to 1
M = (M + M.T) / 2.0
np.fill_diagonal(M, 1.0)

# IDs used by the consensus matrix rows/columns
union_ids = np.asarray(diag['union_ids'])

# --- Final labels and their corresponding IDs ---
# IMPORTANT:
# final_labels must be aligned to the same IDs/order as union_ids.
# This assumes you have final_metrics['final_ids'] available.
final_ids = np.asarray(final_metrics['data']['src_subject_id'])
final_labels = np.asarray(final_metrics['final_labels'])

# --- Basic sanity checks ---
print("M shape:", M.shape)
print("Number of union_ids:", len(union_ids))
print("Number of final_ids:", len(final_ids))
print("Number of final_labels:", len(final_labels))

assert M.shape[0] == M.shape[1], "Consensus matrix must be square."
assert M.shape[0] == len(union_ids), "M rows/columns must match union_ids."
assert len(final_ids) == len(final_labels), "final_ids and final_labels must have the same length."

# --- Align final labels to union_ids ---
label_by_id = dict(zip(final_ids, final_labels))

missing_ids = [uid for uid in union_ids if uid not in label_by_id]

if missing_ids:
    print(f"Warning: {len(missing_ids)} union_ids do not have a final label.")
    print("First missing IDs:", missing_ids[:10])

# Use -1 for missing labels so they sort first.
# Change this if you prefer to drop missing IDs instead.
labels_aligned = np.asarray([
    label_by_id.get(uid, -1)
    for uid in union_ids
])

# --- Sort consensus matrix by aligned final labels ---
order = np.lexsort((np.arange(len(labels_aligned)), labels_aligned))

M_ord = M[np.ix_(order, order)]
labels_ord = labels_aligned[order]
union_ids_ord = union_ids[order]

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(M_ord, aspect='auto', vmin=0, vmax=1)
ax.set_title("Consensus matrix sorted by final labels")
fig.colorbar(im, ax=ax, label="Consensus")

ax.set_xlabel("Samples sorted by final label")
ax.set_ylabel("Samples sorted by final label")

fig.tight_layout()
for ext in ("png", "pdf"):
    output_path = plots_dir_path / f"consensus_matrix_sorted_by_final_labels.{ext}"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print("Saved consensus matrix to:", output_path)
plt.show()

In [ ]:
M = final_metrics['final_stability_SUM_MAT_full']['consensus']
M = np.asarray(M, float)
iu = np.triu_indices_from(M, k=1)
vals = M[iu]

print("fraction <0.1:", np.mean(vals < 0.1))
print("fraction >0.9:", np.mean(vals > 0.9))
print("PAC (0.1..0.9):", np.mean((vals > 0.1) & (vals < 0.9)))
print("mean:", np.mean(vals), "median:", np.median(vals))


## tSNE plots

In [ ]:
final_metrics.keys()


In [ ]:
from Utils import plot_latent_embeddings

for run_name in sorted(dimred_runs):
    if run_name not in dimred_runs:
        print(f'{run_name}: run not found in dimred_runs, skipping component/t-SNE plots.')
        continue
    run_payload = dimred_runs[run_name]
    latent_plot_dir = Path(run_payload['plots_dir'])
    if run_name != best_dimred_option:
        latent_plot_dir = latent_plot_dir / 'alternative_results'
    print(f'\n=== {run_name}: applied-representation/PCA/t-SNE plots ===')
    fig = plot_latent_embeddings(
        run_name,
        run_payload['final_metrics'],
        out_dir=latent_plot_dir,
        file_prefix=run_name,
    )
    if fig is not None:
        plt.show()


## Differences features

### Differences in original features

In [ ]:
final_metrics.keys()


In [ ]:
import glob
import os
import pandas as pd
import importlib
from IPython.display import Image, display
import Utils
Utils = importlib.reload(Utils)
from Utils import make_chr_cc_feature_boxplots

# Apply the learned CHR preprocessing to baseline discovery CC so the
# controls live in the same feature space as the clustered CHR sample.
preproc_artifact = final_metrics['preprocessing_details']

cc_df_discovery = cleaned_discovery_CC.copy()
ae_data_cc_discovery, subject_id_list_cc_discovery, dict_final_cc = apply_preprocessing_to_new_data(
    cc_df_discovery,
    meta,
    preproc_artifact,
    subject_id_column='src_subject_id'
)

# Reproduce the simpleclust merged schema: Modality__feature.
# Passing only the first modality leaves no columns in common with
# final_metrics['data'], whose features were prefixed during training.
cc_preprocessed = None
for cc_modality in preproc_artifact['modalities_in_output']:
    if cc_modality not in dict_final_cc:
        continue
    cc_modality_df = dict_final_cc[cc_modality].copy()
    cc_feature_columns = [
        column for column in cc_modality_df.columns
        if column != 'src_subject_id'
    ]
    cc_modality_df = cc_modality_df.rename(columns={
        column: f'{cc_modality}__{column}'
        for column in cc_feature_columns
    })
    cc_preprocessed = (
        cc_modality_df
        if cc_preprocessed is None
        else cc_preprocessed.merge(
            cc_modality_df,
            on='src_subject_id',
            how='inner',
            validate='one_to_one'
        )
    )

if cc_preprocessed is None or cc_preprocessed.empty:
    raise ValueError('No aligned discovery CC modalities were available after preprocessing.')

training_feature_order = [
    column for column in final_metrics['data'].columns
    if column != 'src_subject_id'
]
missing_cc_features = [
    column for column in training_feature_order
    if column not in cc_preprocessed.columns
]
if missing_cc_features:
    raise ValueError(
        f'CC preprocessing is missing {len(missing_cc_features)} training features; '
        f'examples: {missing_cc_features[:10]}'
    )
cc_preprocessed = cc_preprocessed[
    ['src_subject_id'] + training_feature_order
]
print('Discovery CC preprocessing complete:', cc_preprocessed.shape)

feature_out_dir = os.path.join(plots_dir, 'merged_feature_differences_chr_vs_cc')
os.makedirs(feature_out_dir, exist_ok=True)

baseline_feature_stats = make_chr_cc_feature_boxplots(
    chr_df=final_metrics['data'],
    clusters=final_metrics['final_labels'],
    cc_df=cc_preprocessed,
    top_k=30,
    out_dir=feature_out_dir,
    file_prefix='baseline_chr_vs_cc',
    cc_label='CC',
    title_prefix='Baseline top features by simple-cluster subgroup'
)

baseline_feature_stats.to_csv(
    os.path.join(feature_out_dir, 'baseline_chr_vs_cc_feature_stats.csv'),
    index=False
)
print('Saved baseline CHR-vs-CC feature stats to:', os.path.join(feature_out_dir, 'baseline_chr_vs_cc_feature_stats.csv'))

for png_path in sorted(glob.glob(os.path.join(feature_out_dir, 'baseline_chr_vs_cc_boxplots_page_*.png'))):
    display(Image(filename=png_path))



In [ ]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

try:
    from Utils import display_feature_name, safe_name
except Exception:
    def display_feature_name(feature):
        return str(feature).split("__", 1)[-1]
    def safe_name(value):
        return str(value).replace("/", "_").replace(" ", "_")


def _bh_fdr_original_features(p_values):
    p_values = pd.Series(p_values, dtype="float64")
    q_values = pd.Series(np.nan, index=p_values.index, dtype="float64")
    valid = p_values.dropna()
    if valid.empty:
        return q_values
    ordered = valid.sort_values()
    n_tests = float(len(ordered))
    adjusted = ordered.to_numpy() * n_tests / np.arange(1, len(ordered) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    q_values.loc[ordered.index] = np.clip(adjusted, 0, 1)
    return q_values


def _format_fdr_q(q_value):
    if pd.isna(q_value):
        return ""
    if q_value < 0.001:
        return "<0.001"
    if q_value < 0.01:
        return "<0.01"
    if q_value < 0.05:
        return "<0.05"
    return f"{q_value:.2g}"


def _cohens_d(values_a, values_b):
    values_a = pd.Series(values_a, dtype="float64").dropna()
    values_b = pd.Series(values_b, dtype="float64").dropna()
    n_a, n_b = len(values_a), len(values_b)
    if n_a < 2 or n_b < 2:
        return np.nan
    var_a = values_a.var(ddof=1)
    var_b = values_b.var(ddof=1)
    pooled_var = ((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2)
    if not np.isfinite(pooled_var) or pooled_var <= 0:
        return np.nan
    return float((values_a.mean() - values_b.mean()) / np.sqrt(pooled_var))


def make_original_feature_pairwise_tests(chr_df, clusters, ranked_stats, subject_id_column="src_subject_id"):
    features = ranked_stats["feature"].drop_duplicates().tolist()
    cluster_values = pd.Series(np.asarray(clusters).reshape(-1), name="group").astype(str)
    group_order = sorted(cluster_values.dropna().unique().tolist())
    rows = []

    for feature_rank, feature in enumerate(features, start=1):
        if feature not in chr_df.columns:
            continue
        chr_values = pd.to_numeric(chr_df[feature], errors="coerce")
        plot_df = pd.DataFrame({"value": chr_values, "group": cluster_values}).dropna(subset=["value", "group"])
        if plot_df.empty:
            continue
        for group_a, group_b in combinations(group_order, 2):
            values_a = plot_df.loc[plot_df["group"].eq(group_a), "value"].dropna().astype(float)
            values_b = plot_df.loc[plot_df["group"].eq(group_b), "value"].dropna().astype(float)
            statistic, p_value = np.nan, np.nan
            if len(values_a) > 0 and len(values_b) > 0:
                try:
                    statistic, p_value = mannwhitneyu(values_a, values_b, alternative="two-sided")
                except ValueError:
                    pass
            rows.append({
                "feature_rank": feature_rank,
                "feature": feature,
                "display_feature": display_feature_name(feature),
                "comparison": f"{group_a} vs {group_b}",
                "group_a": group_a,
                "group_b": group_b,
                "n_a": int(values_a.notna().sum()),
                "n_b": int(values_b.notna().sum()),
                "median_a": values_a.median() if len(values_a) else np.nan,
                "median_b": values_b.median() if len(values_b) else np.nan,
                "median_difference_a_minus_b": (
                    values_a.median() - values_b.median() if len(values_a) and len(values_b) else np.nan
                ),
                "mann_whitney_u": statistic,
                "effect_size": _cohens_d(values_a, values_b),
                "p_value": p_value,
            })

    out = pd.DataFrame(rows)
    if not out.empty:
        out["pairwise_q_value_fdr"] = _bh_fdr_original_features(out["p_value"])
        out = out.sort_values(
            ["pairwise_q_value_fdr", "p_value", "feature_rank", "comparison"],
            na_position="last",
        ).reset_index(drop=True)
    return out


def plot_original_feature_pairwise_tiles(pairwise_df, ranked_stats, out_dir, file_prefix="baseline_chr_vs_cc", top_n=30):
    if pairwise_df.empty:
        return None
    top_features = ranked_stats["feature"].drop_duplicates().head(top_n).tolist()
    plot_df = pairwise_df[pairwise_df["feature"].isin(top_features)].dropna(subset=["effect_size"]).copy()
    if plot_df.empty:
        return None
    effect = plot_df.pivot_table(
        index="display_feature",
        columns="comparison",
        values="effect_size",
        aggfunc="first",
    )
    q_values = plot_df.pivot_table(
        index="display_feature",
        columns="comparison",
        values="pairwise_q_value_fdr",
        aggfunc="min",
    )
    feature_order = [display_feature_name(feature) for feature in top_features if display_feature_name(feature) in effect.index]
    effect = effect.reindex(feature_order)
    q_values = q_values.reindex(feature_order)
    annot = q_values.applymap(_format_fdr_q)
    vmax = np.nanmax(np.abs(effect.to_numpy(dtype=float))) if effect.notna().any().any() else 1.0
    vmax = max(vmax, 1e-6)
    fig_height = max(5, 0.28 * len(effect) + 1.8)
    fig_width = max(7, 0.70 * effect.shape[1] + 3.2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    sns.heatmap(
        effect,
        cmap="vlag",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        annot=annot,
        fmt="",
        linewidths=0.35,
        linecolor="white",
        cbar_kws={"label": "Effect size: Cohen's d (group A - group B)"},
        ax=ax,
    )
    ax.set_title("Original features: CHR subgroup pairwise post-hoc tests")
    ax.set_xlabel("Pairwise group comparison; tile text is FDR q")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    out_dir = Path(out_dir)
    png_path = out_dir / f"{safe_name(file_prefix)}_all_pairwise_effect_size_tiles.png"
    pdf_path = out_dir / f"{safe_name(file_prefix)}_all_pairwise_effect_size_tiles.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Saved original-feature pairwise effect-size tile plot to:", png_path)
    print("Saved original-feature pairwise effect-size tile plot to:", pdf_path)
    plt.show()
    return fig
baseline_feature_pairwise_stats = make_original_feature_pairwise_tests(
    chr_df=final_metrics["data"],
    clusters=final_metrics["final_labels"],
    ranked_stats=baseline_feature_stats,
)

baseline_pairwise_path = Path(feature_out_dir) / "baseline_chr_vs_cc_all_pairwise_feature_tests.csv"
baseline_feature_pairwise_stats.to_csv(baseline_pairwise_path, index=False)
print("Saved baseline original-feature CHR-subgroup pairwise post-hoc tests to:", baseline_pairwise_path)
display(baseline_feature_pairwise_stats.head(30))

plot_original_feature_pairwise_tiles(
    pairwise_df=baseline_feature_pairwise_stats,
    ranked_stats=baseline_feature_stats,
    out_dir=feature_out_dir,
    file_prefix="baseline_chr_vs_cc",
    top_n=30,
)


#### Profile plots subgroups

In [ ]:
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import plot_subgroup_feature_profiles_by_modality

simple_profile_summaries = plot_subgroup_feature_profiles_by_modality(
    data_by_modality={"Simple clustering": final_metrics["data"]},
    labels_by_modality={"Simple clustering": final_metrics["final_labels"]},
    control_data_by_modality={"Simple clustering": cc_preprocessed},
    plots_dir=plots_dir,
    subject_id_column="src_subject_id",
    sample_label="discovery",
    subgroup_label="Subgroup",
    control_label="Community controls",
    control_color=CC_COLOR,
    plot_kinds=("line", "dot", "heatmap"),
    show=True,
)


### Differences extra features

In [ ]:
import theme
from IPython.display import display
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy.stats import chi2_contingency, fisher_exact, kruskal, mannwhitneyu

try:
    from Utils import LABEL_SPECS, display_feature_name
except Exception:
    LABEL_SPECS = []
    def display_feature_name(feature):
        return str(feature).split("__", 1)[-1]


def _bh_fdr(p_values):
    p_values = pd.Series(p_values, dtype="float64")
    q_values = pd.Series(np.nan, index=p_values.index, dtype="float64")
    valid = p_values.dropna()
    if valid.empty:
        return q_values
    ordered = valid.sort_values()
    n_tests = float(len(ordered))
    adjusted = ordered.to_numpy() * n_tests / np.arange(1, len(ordered) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    q_values.loc[ordered.index] = np.clip(adjusted, 0, 1)
    return q_values


def _clean_compare_series(series):
    return series.replace(["nan", "NaN", "None", "NULL", "null", ""], np.nan)


def _ordinal_label_maps_from_utils(label_specs):
    ordinal_maps = []
    for spec in label_specs or []:
        if "mapping" in spec:
            mapping = spec["mapping"]
        elif spec.get("fill_middle"):
            first, last = spec["first"], spec["last"]
            mapping = {first: spec["first_label"], last: spec["last_label"]}
            for code in range(first + 1, last):
                mapping[code] = str(code)
        else:
            continue
        reverse_map = {str(label).strip(): code for code, label in mapping.items()}
        ordinal_maps.append((set(reverse_map), reverse_map))
    return ordinal_maps


ORDINAL_LABEL_MAPS = _ordinal_label_maps_from_utils(LABEL_SPECS)


def _coerce_compare_variable(series):
    cleaned = _clean_compare_series(series)
    nonmissing = cleaned.notna()
    if nonmissing.sum() == 0:
        return cleaned, "categorical"

    numeric = pd.to_numeric(cleaned, errors="coerce")
    if numeric.loc[nonmissing].notna().all():
        return numeric, "numeric"

    observed_labels = set(cleaned.loc[nonmissing].astype(str).str.strip().unique())
    for label_set, reverse_map in ORDINAL_LABEL_MAPS:
        if observed_labels.issubset(label_set):
            mapped = cleaned.astype("object").map(
                lambda value: reverse_map.get(str(value).strip(), np.nan) if pd.notna(value) else np.nan
            )
            return pd.to_numeric(mapped, errors="coerce"), "ordinal_mapped"

    return cleaned.astype("object"), "categorical"


def _cramers_v_from_contingency(contingency):
    if contingency.empty:
        return np.nan
    try:
        chi2_stat, _, _, _ = chi2_contingency(contingency)
    except ValueError:
        return np.nan
    n = contingency.to_numpy().sum()
    if n <= 0:
        return np.nan
    denom = n * max(min(contingency.shape) - 1, 1)
    return float(np.sqrt(chi2_stat / denom)) if denom > 0 else np.nan


def _format_fdr_q(q_value):
    if pd.isna(q_value):
        return ""
    if q_value < 0.001:
        return "<0.001"
    if q_value < 0.01:
        return "<0.01"
    if q_value < 0.05:
        return "<0.05"
    return f"{q_value:.2g}"


def _cohens_d(values_a, values_b):
    values_a = pd.Series(values_a, dtype="float64").dropna()
    values_b = pd.Series(values_b, dtype="float64").dropna()
    n_a, n_b = len(values_a), len(values_b)
    if n_a < 2 or n_b < 2:
        return np.nan
    var_a = values_a.var(ddof=1)
    var_b = values_b.var(ddof=1)
    pooled_var = ((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2)
    if not np.isfinite(pooled_var) or pooled_var <= 0:
        return np.nan
    return float((values_a.mean() - values_b.mean()) / np.sqrt(pooled_var))


def _pairwise_numeric_tests(values, groups, variable, display_name, variable_type, group_order):
    rows = []
    for group_a, group_b in combinations(group_order, 2):
        values_a = values[groups.eq(group_a)].dropna().astype(float)
        values_b = values[groups.eq(group_b)].dropna().astype(float)
        statistic, p_value = np.nan, np.nan
        if len(values_a) > 0 and len(values_b) > 0:
            try:
                statistic, p_value = mannwhitneyu(values_a, values_b, alternative="two-sided")
            except ValueError:
                pass
        rows.append({
            "feature": variable,
            "display_feature": display_name,
            "variable_type": variable_type,
            "test": "Mann-Whitney U",
            "comparison": f"Subgroup {group_a} vs Subgroup {group_b}",
            "group_a": group_a,
            "group_b": group_b,
            "n_a": int(values_a.notna().sum()),
            "n_b": int(values_b.notna().sum()),
            "median_a": values_a.median() if len(values_a) else np.nan,
            "median_b": values_b.median() if len(values_b) else np.nan,
            "median_difference_a_minus_b": (
                values_a.median() - values_b.median() if len(values_a) and len(values_b) else np.nan
            ),
            "statistic": statistic,
            "effect_size": _cohens_d(values_a, values_b),
            "p_value": p_value,
        })
    return rows


def _pairwise_categorical_tests(values, groups, variable, display_name, variable_type, group_order):
    rows = []
    for group_a, group_b in combinations(group_order, 2):
        pair_df = pd.DataFrame({"value": values, "group": groups})
        pair_df = pair_df[pair_df["group"].isin([group_a, group_b])].dropna()
        statistic, p_value, test_name, effect_size = np.nan, np.nan, "Chi-square", np.nan
        if pair_df["value"].nunique() >= 2 and pair_df["group"].nunique() == 2:
            contingency = pd.crosstab(pair_df["value"], pair_df["group"])
            effect_size = _cramers_v_from_contingency(contingency)
            try:
                if contingency.shape == (2, 2):
                    _, p_value = fisher_exact(contingency.to_numpy())
                    test_name = "Fisher exact"
                else:
                    statistic, p_value, _, _ = chi2_contingency(contingency)
            except ValueError:
                pass
        rows.append({
            "feature": variable,
            "display_feature": display_name,
            "variable_type": variable_type,
            "test": test_name,
            "comparison": f"Subgroup {group_a} vs Subgroup {group_b}",
            "group_a": group_a,
            "group_b": group_b,
            "n_a": int((pair_df["group"] == group_a).sum()),
            "n_b": int((pair_df["group"] == group_b).sum()),
            "median_a": np.nan,
            "median_b": np.nan,
            "median_difference_a_minus_b": np.nan,
            "statistic": statistic,
            "effect_size": effect_size,
            "p_value": p_value,
        })
    return rows


def plot_top_feature_profile_dotplot(stats_df, long_df, out_dir, file_prefix, title, top_n=None, rows_per_page=24):
    if stats_df.empty or long_df.empty:
        return []
    ordered_features = stats_df.sort_values(["q_value_fdr", "p_value"], na_position="last")["feature"].tolist()
    if top_n is not None:
        ordered_features = ordered_features[:int(top_n)]
    plot_df = long_df[long_df["feature"].isin(ordered_features)].copy()
    if plot_df.empty:
        return []
    selected_rows = []
    for feature in ordered_features:
        feature_df = plot_df[plot_df["feature"].eq(feature)]
        if feature_df.empty:
            continue
        if feature_df["variable_type"].iloc[0] == "categorical":
            row_label = (
                feature_df.groupby("row_label")["standardized_difference"]
                .apply(lambda values: values.abs().max())
                .sort_values(ascending=False)
                .index[0]
            )
            feature_df = feature_df[feature_df["row_label"].eq(row_label)]
        selected_rows.append(feature_df)
    if not selected_rows:
        return []
    plot_df = pd.concat(selected_rows, ignore_index=True)
    q_lookup = stats_df.set_index("feature")["q_value_fdr"].to_dict()
    plot_df["feature_label"] = plot_df.apply(
        lambda row: f"{row['row_label']} (q={_format_fdr_q(q_lookup.get(row['feature'], np.nan))})",
        axis=1,
    )
    feature_order = plot_df.drop_duplicates("feature_label")["feature_label"].tolist()
    pages = [feature_order]
    figures = []
    for page_index, page_features in enumerate(pages, start=1):
        page_df = plot_df[plot_df["feature_label"].isin(page_features)].copy()
        page_order = page_features[::-1]
        page_df["feature_label"] = pd.Categorical(page_df["feature_label"], categories=page_order, ordered=True)
        y_codes = page_df["feature_label"].cat.codes.to_numpy(dtype=float)
        effect_values = page_df["standardized_difference"].to_numpy(dtype=float)
        vmax = np.nanmax(np.abs(effect_values)) if np.isfinite(effect_values).any() else 1.0
        vmax = max(vmax, 1e-6)
        subgroup_values = sorted(page_df["subgroup"].dropna().unique().tolist())
        marker_cycle = ["o", "s", "^", "D", "X", "P"]
        marker_map = {subgroup: marker_cycle[i % len(marker_cycle)] for i, subgroup in enumerate(subgroup_values)}
        fig_height = max(7, 0.30 * len(page_order) + 2.0)
        fig, ax = plt.subplots(figsize=(9.5, fig_height))
        scatter = None
        for subgroup in subgroup_values:
            mask = page_df["subgroup"].eq(subgroup).to_numpy()
            scatter = ax.scatter(
                effect_values[mask],
                y_codes[mask],
                c=effect_values[mask],
                cmap="vlag",
                vmin=-vmax,
                vmax=vmax,
                marker=marker_map[subgroup],
                s=58,
                edgecolor="black",
                linewidth=0.25,
                label=f"Subgroup {subgroup}",
            )
        ax.axvline(0, color=theme.THEME["muted"], linewidth=0.8)
        page_suffix = f" page {page_index}/{len(pages)}" if len(pages) > 1 else ""
        ax.set_title(f"{title}{page_suffix}")
        ax.set_xlabel("Effect size: standardized subgroup difference from labelled discovery sample")
        ax.set_ylabel("")
        ax.set_yticks(np.arange(len(page_order)))
        ax.set_yticklabels(page_order)
        ax.legend(title="Subgroup", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        if scatter is not None:
            cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
            cbar.set_label("Effect size")
        fig.tight_layout()
        suffix = "all_feature_profile_dotplot" if top_n is None else "top_feature_profile_dotplot"
        page_token = f"_page_{page_index:02d}" if len(pages) > 1 else ""
        png_path = Path(out_dir) / f"{file_prefix}_{suffix}{page_token}.png"
        pdf_path = Path(out_dir) / f"{file_prefix}_{suffix}{page_token}.pdf"
        fig.savefig(png_path, dpi=300, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        print("Saved feature profile dot plot to:", png_path)
        print("Saved feature profile dot plot to:", pdf_path)
        plt.show()
        figures.append(fig)
    return figures
def plot_pairwise_fdr_tiles(pairwise_df, stats_df, out_dir, file_prefix, title, top_n=None, rows_per_page=24):
    if pairwise_df.empty or stats_df.empty:
        return []
    ordered_features = stats_df.sort_values(["q_value_fdr", "p_value"], na_position="last")["feature"].tolist()
    if top_n is not None:
        ordered_features = ordered_features[:int(top_n)]
    plot_df = pairwise_df[pairwise_df["feature"].isin(ordered_features)].copy()
    plot_df = plot_df.dropna(subset=["effect_size"])
    if plot_df.empty:
        return []
    plot_df["feature_label"] = plot_df["display_feature"]
    feature_order = [display_feature_name(feature) for feature in ordered_features if display_feature_name(feature) in set(plot_df["feature_label"])]
    pages = [feature_order]
    figures = []
    for page_index, page_features in enumerate(pages, start=1):
        page_df = plot_df[plot_df["feature_label"].isin(page_features)].copy()
        effect = page_df.pivot_table(
            index="feature_label",
            columns="comparison",
            values="effect_size",
            aggfunc="first",
        ).reindex(page_features)
        q_values = page_df.pivot_table(
            index="feature_label",
            columns="comparison",
            values="pairwise_q_value_fdr",
            aggfunc="min",
        ).reindex(page_features)
        annot = q_values.applymap(_format_fdr_q)
        vmax = np.nanmax(np.abs(effect.to_numpy(dtype=float))) if effect.notna().any().any() else 1.0
        vmax = max(vmax, 1e-6)
        fig_height = max(7, 0.28 * len(effect) + 2.0)
        fig_width = max(7.5, 0.72 * effect.shape[1] + 3.2)
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        sns.heatmap(
            effect,
            cmap="vlag",
            center=0,
            vmin=-vmax,
            vmax=vmax,
            annot=annot,
            fmt="",
            linewidths=0.35,
            linecolor="white",
            cbar_kws={"label": "Effect size: Cohen's d; Cramer's V for categorical"},
            ax=ax,
        )
        page_suffix = f" page {page_index}/{len(pages)}" if len(pages) > 1 else ""
        ax.set_title(f"{title}{page_suffix}")
        ax.set_xlabel("Pairwise subgroup comparison; tile text is FDR q")
        ax.set_ylabel("")
        ax.tick_params(axis="x", rotation=35)
        fig.tight_layout()
        suffix = "pairwise_effect_size_tiles" if top_n is None else "top_pairwise_effect_size_tiles"
        page_token = f"_page_{page_index:02d}" if len(pages) > 1 else ""
        png_path = Path(out_dir) / f"{file_prefix}_{suffix}{page_token}.png"
        pdf_path = Path(out_dir) / f"{file_prefix}_{suffix}{page_token}.pdf"
        fig.savefig(png_path, dpi=300, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        print("Saved pairwise effect-size tile plot to:", png_path)
        print("Saved pairwise effect-size tile plot to:", pdf_path)
        plt.show()
        figures.append(fig)
    return figures
def compare_extra_features_by_cluster(
    source_df,
    final_data,
    final_labels,
    meta,
    modality="compare_clusters",
    subject_id_column="src_subject_id",
):
    compare_vars = (
        meta.loc[meta["Modality"].eq(modality), "ElementName"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )
    available_vars = [var for var in compare_vars if var in source_df.columns]
    missing_vars = [var for var in compare_vars if var not in source_df.columns]
    if not available_vars:
        raise ValueError(
            f"No {modality!r} metadata variables were found in the source dataframe. "
            f"Examples from metadata: {compare_vars[:8]}"
        )

    labels_df = pd.DataFrame({
        subject_id_column: final_data[subject_id_column].astype(str).to_numpy(),
        "simpleclust_subgroup": np.asarray(final_labels),
    })
    analysis_df = (
        source_df[[subject_id_column] + available_vars]
        .copy()
        .assign(**{subject_id_column: lambda df: df[subject_id_column].astype(str)})
        .merge(labels_df, on=subject_id_column, how="inner", validate="one_to_one")
    )
    if analysis_df.empty:
        raise ValueError("No labelled discovery subjects were available after merging extra features by subject ID.")

    subgroup_order = sorted(analysis_df["simpleclust_subgroup"].dropna().unique())
    stats_rows = []
    heatmap_rows = []
    long_rows = []
    pairwise_rows = []

    for variable in available_vars:
        raw = _clean_compare_series(analysis_df[variable])
        coerced_values, variable_type = _coerce_compare_variable(raw)
        display_name = display_feature_name(variable)

        if variable_type in {"numeric", "ordinal_mapped"}:
            values = pd.to_numeric(coerced_values, errors="coerce")
            test_df = pd.DataFrame({"value": values, "subgroup": analysis_df["simpleclust_subgroup"]}).dropna()
            groups = [group["value"].to_numpy() for _, group in test_df.groupby("subgroup") if len(group) > 0]
            if len(groups) >= 2 and sum(len(group) > 1 for group in groups) >= 2:
                stat, p_value = kruskal(*groups)
            else:
                stat, p_value = np.nan, np.nan

            overall_mean = values.mean(skipna=True)
            overall_sd = values.std(skipna=True, ddof=0)
            if not np.isfinite(overall_sd) or overall_sd == 0:
                overall_sd = 1.0

            row = {"feature": variable, "display_feature": display_name, "row_label": display_name}
            for subgroup in subgroup_order:
                subgroup_values = values[analysis_df["simpleclust_subgroup"].eq(subgroup)]
                row[f"Subgroup {subgroup}"] = (subgroup_values.mean(skipna=True) - overall_mean) / overall_sd
                long_rows.append({
                    "feature": variable,
                    "display_feature": display_name,
                    "row_label": display_name,
                    "variable_type": variable_type,
                    "subgroup": subgroup,
                    "n": int(subgroup_values.notna().sum()),
                    "mean_or_proportion": subgroup_values.mean(skipna=True),
                    "standardized_difference": row[f"Subgroup {subgroup}"],
                })
            heatmap_rows.append(row)
            pairwise_rows.extend(
                _pairwise_numeric_tests(
                    values=values,
                    groups=analysis_df["simpleclust_subgroup"],
                    variable=variable,
                    display_name=display_name,
                    variable_type=variable_type,
                    group_order=subgroup_order,
                )
            )

            stats_rows.append({
                "feature": variable,
                "display_feature": display_name,
                "variable_type": variable_type,
                "test": "Kruskal-Wallis",
                "statistic": stat,
                "p_value": p_value,
                "n": int(values.notna().sum()),
                "missing": int(values.isna().sum()),
                "levels_or_categories": np.nan,
            })
            continue

        categorical = coerced_values.astype("object").where(raw.notna(), np.nan)
        test_df = pd.DataFrame({"value": categorical, "subgroup": analysis_df["simpleclust_subgroup"]}).dropna()
        if test_df.empty or test_df["value"].nunique() < 2 or test_df["subgroup"].nunique() < 2:
            stat, p_value, test_name = np.nan, np.nan, "Chi-square"
        else:
            contingency = pd.crosstab(test_df["value"], test_df["subgroup"])
            if contingency.shape == (2, 2):
                _, p_value = fisher_exact(contingency.to_numpy())
                stat, test_name = np.nan, "Fisher exact"
            else:
                stat, p_value, _, _ = chi2_contingency(contingency)
                test_name = "Chi-square"

        stats_rows.append({
            "feature": variable,
            "display_feature": display_name,
            "variable_type": variable_type,
            "test": test_name,
            "statistic": stat,
            "p_value": p_value,
            "n": int(categorical.notna().sum()),
            "missing": int(categorical.isna().sum()),
            "levels_or_categories": int(categorical.nunique(dropna=True)),
        })

        overall_props = categorical.value_counts(normalize=True, dropna=True)
        for category in sorted(overall_props.index, key=lambda value: str(value)):
            category_mask = categorical.eq(category)
            overall_p = overall_props.loc[category]
            denom = np.sqrt(max(overall_p * (1 - overall_p), 1e-6))
            row_label = f"{display_name} = {category}"
            row = {"feature": variable, "display_feature": display_name, "row_label": row_label}
            for subgroup in subgroup_order:
                subgroup_mask = analysis_df["simpleclust_subgroup"].eq(subgroup)
                subgroup_n = int((subgroup_mask & categorical.notna()).sum())
                subgroup_p = category_mask[subgroup_mask & categorical.notna()].mean() if subgroup_n else np.nan
                row[f"Subgroup {subgroup}"] = (subgroup_p - overall_p) / denom if pd.notna(subgroup_p) else np.nan
                long_rows.append({
                    "feature": variable,
                    "display_feature": display_name,
                    "row_label": row_label,
                    "variable_type": variable_type,
                    "subgroup": subgroup,
                    "n": subgroup_n,
                    "mean_or_proportion": subgroup_p,
                    "standardized_difference": row[f"Subgroup {subgroup}"],
                })
            heatmap_rows.append(row)

        pairwise_rows.extend(
            _pairwise_categorical_tests(
                values=categorical,
                groups=analysis_df["simpleclust_subgroup"],
                variable=variable,
                display_name=display_name,
                variable_type=variable_type,
                group_order=subgroup_order,
            )
        )

    stats_df = pd.DataFrame(stats_rows)
    stats_df["q_value_fdr"] = _bh_fdr(stats_df["p_value"])
    stats_df["significant_fdr_0_05"] = stats_df["q_value_fdr"].lt(0.05)
    stats_df = stats_df.sort_values(["q_value_fdr", "p_value", "display_feature"], na_position="last").reset_index(drop=True)

    heatmap_df = pd.DataFrame(heatmap_rows)
    if not heatmap_df.empty:
        ordering = stats_df.set_index("feature")["q_value_fdr"].to_dict()
        heatmap_df["_q_value_fdr"] = heatmap_df["feature"].map(ordering)
        heatmap_df = heatmap_df.sort_values(["_q_value_fdr", "display_feature", "row_label"], na_position="last")
        q_lookup = stats_df.set_index("feature")["q_value_fdr"].to_dict()
        p_lookup = stats_df.set_index("feature")["p_value"].to_dict()
        heatmap_df["row_label_with_q"] = heatmap_df.apply(
            lambda row: f"{row['row_label']}  (q={q_lookup.get(row['feature'], np.nan):.3g})"
            if pd.notna(q_lookup.get(row["feature"], np.nan))
            else row["row_label"],
            axis=1,
        )
        heatmap_df["p_value"] = heatmap_df["feature"].map(p_lookup)
        heatmap_df["q_value_fdr"] = heatmap_df["feature"].map(q_lookup)

    long_df = pd.DataFrame(long_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)
    if not pairwise_df.empty:
        pairwise_df["pairwise_q_value_fdr"] = _bh_fdr(pairwise_df["p_value"])
        pairwise_df = pairwise_df.sort_values(
            ["pairwise_q_value_fdr", "p_value", "display_feature", "comparison"],
            na_position="last",
        ).reset_index(drop=True)
    return stats_df, heatmap_df, long_df, pairwise_df, missing_vars


compare_extra_out_dir = Path(plots_dir) / "differences_extra_features"
compare_extra_out_dir.mkdir(parents=True, exist_ok=True)

extra_feature_stats, extra_feature_heatmap_df, extra_feature_long_df, extra_feature_pairwise_df, missing_compare_vars = compare_extra_features_by_cluster(
    source_df=discovery_data,
    final_data=final_metrics["data"],
    final_labels=final_metrics["final_labels"],
    meta=meta,
    modality="compare_clusters",
    subject_id_column="src_subject_id",
)

stats_path = compare_extra_out_dir / "compare_clusters_extra_feature_tests.csv"
pairwise_path = compare_extra_out_dir / "compare_clusters_extra_feature_pairwise_tests.csv"
long_path = compare_extra_out_dir / "compare_clusters_extra_feature_subgroup_profiles.csv"
heatmap_matrix_path = compare_extra_out_dir / "compare_clusters_extra_feature_heatmap_matrix.csv"
extra_feature_stats.to_csv(stats_path, index=False)
extra_feature_pairwise_df.to_csv(pairwise_path, index=False)
extra_feature_long_df.to_csv(long_path, index=False)
extra_feature_heatmap_df.to_csv(heatmap_matrix_path, index=False)

print(f"Compared {extra_feature_stats.shape[0]} compare_clusters variables across Simpleclust subgroups.")
if missing_compare_vars:
    print(f"Metadata variables not found in discovery_data ({len(missing_compare_vars)}): {missing_compare_vars[:10]}")
print("Saved extra-feature test table to:", stats_path)
print("Saved extra-feature pairwise post-hoc table to:", pairwise_path)
print("Saved extra-feature subgroup profile table to:", long_path)

display(extra_feature_stats.head(30))
display(extra_feature_pairwise_df.head(30))

plot_top_feature_profile_dotplot(
    stats_df=extra_feature_stats,
    long_df=extra_feature_long_df,
    out_dir=compare_extra_out_dir,
    file_prefix="compare_clusters_extra_features",
    title="Compare-cluster variables by Simpleclust subgroup",
    top_n=None,
)

plot_pairwise_fdr_tiles(
    pairwise_df=extra_feature_pairwise_df,
    stats_df=extra_feature_stats,
    out_dir=compare_extra_out_dir,
    file_prefix="compare_clusters_extra_features",
    title="Pairwise post-hoc tests for compare-cluster variables",
    top_n=None,
)

print(
    "Full compare_clusters heatmap was not rendered because the all-variable effect-size plots are more readable. "
    "The full matrix is still saved to:",
    heatmap_matrix_path,
)


In [ ]:

from IPython.display import Image, display
from pathlib import Path
import glob
import importlib
import json
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

import Utils
Utils = importlib.reload(Utils)
from Utils import (
    apply_preprocessing_to_new_data,
    make_chr_cc_feature_boxplots,
    plot_subgroup_feature_profiles_by_modality,
    safe_name,
)


def _alternative_run_names():
    return [
        run_name
        for run_name in sorted(dimred_runs)
        if run_name != best_dimred_option
    ]


def plot_dimred_consensus_matrix(run_name, run_metrics, out_dir=None):
    diag = run_metrics.get('final_stability_SUM_MAT_full', {})
    if 'consensus' not in diag:
        print(f'{run_name}: no final consensus matrix available.')
        return None

    M = np.asarray(diag['consensus'], dtype=float)
    if M.ndim != 2 or M.shape[0] != M.shape[1]:
        print(f'{run_name}: final consensus matrix is not square, skipping matrix plot.')
        return None

    M = (M + M.T) / 2.0
    np.fill_diagonal(M, 1.0)
    title_suffix = 'hierarchical order'

    union_ids = np.asarray(diag.get('union_ids', []))
    final_data = run_metrics.get('data')
    final_labels = np.asarray(run_metrics.get('final_labels', []))
    if (
        isinstance(final_data, pd.DataFrame)
        and 'src_subject_id' in final_data.columns
        and len(union_ids) == M.shape[0]
        and len(final_labels) == len(final_data)
    ):
        label_by_id = dict(zip(final_data['src_subject_id'].to_numpy(), final_labels))
        labels_aligned = np.asarray([label_by_id.get(uid, -1) for uid in union_ids])
        order = np.lexsort((np.arange(len(labels_aligned)), labels_aligned))
        title_suffix = 'sorted by final labels'
    else:
        D = 1.0 - M
        order = leaves_list(linkage(squareform(D, checks=False), method='average'))

    M_ord = M[np.ix_(order, order)]

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(M_ord, aspect='auto', vmin=0, vmax=1)
    ax.set_title(f'{run_name}: consensus matrix ({title_suffix})')
    ax.set_xlabel('Samples')
    ax.set_ylabel('Samples')
    fig.colorbar(im, ax=ax, label='Consensus')
    fig.tight_layout()

    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
        for ext in ('png', 'pdf'):
            matrix_path = os.path.join(out_dir, f'{safe_name(run_name)}_consensus_matrix.{ext}')
            fig.savefig(matrix_path, dpi=300, bbox_inches='tight')
            print('Saved matrix plot to:', matrix_path)

    plt.show()
    return fig


def _merge_simpleclust_cc_preprocessed(dict_final_cc, preproc_artifact, training_feature_order):
    cc_preprocessed = None
    for cc_modality in preproc_artifact.get('modalities_in_output', []):
        if cc_modality not in dict_final_cc:
            continue
        cc_modality_df = dict_final_cc[cc_modality].copy()
        cc_feature_columns = [
            column for column in cc_modality_df.columns
            if column != 'src_subject_id'
        ]
        prefixed_df = cc_modality_df.rename(columns={
            column: f'{cc_modality}__{column}'
            for column in cc_feature_columns
        })
        cc_preprocessed = (
            prefixed_df
            if cc_preprocessed is None
            else cc_preprocessed.merge(
                prefixed_df,
                on='src_subject_id',
                how='inner',
                validate='one_to_one',
            )
        )

    if cc_preprocessed is None or cc_preprocessed.empty:
        raise ValueError('No aligned discovery CC modalities were available after preprocessing.')

    missing_features = [
        column for column in training_feature_order
        if column not in cc_preprocessed.columns
    ]
    if missing_features and len(dict_final_cc) == 1:
        # Support older/simple single-modality exports that did not prefix columns.
        only_modality_df = next(iter(dict_final_cc.values())).copy()
        unprefixed_missing = [
            column for column in training_feature_order
            if column not in only_modality_df.columns
        ]
        if not unprefixed_missing:
            cc_preprocessed = only_modality_df
            missing_features = []

    if missing_features:
        raise ValueError(
            f'CC preprocessing is missing {len(missing_features)} training features; '
            f'examples: {missing_features[:10]}'
        )

    return cc_preprocessed[['src_subject_id'] + training_feature_order]


def build_cc_preprocessed_for_dimred_run(run_metrics):
    preproc_artifact = run_metrics.get('preprocessing_details')
    if preproc_artifact is None:
        raise ValueError('Run does not contain preprocessing_details.')

    _, _, dict_final_cc = apply_preprocessing_to_new_data(
        cleaned_discovery_CC.copy(),
        meta,
        preproc_artifact,
        subject_id_column='src_subject_id',
    )
    training_feature_order = [
        column for column in run_metrics['data'].columns
        if column != 'src_subject_id'
    ]
    return _merge_simpleclust_cc_preprocessed(
        dict_final_cc=dict_final_cc,
        preproc_artifact=preproc_artifact,
        training_feature_order=training_feature_order,
    )


def _display_saved_boxplot_pages(out_dir, file_prefix):
    pattern = os.path.join(out_dir, f'{safe_name(file_prefix)}_boxplots_page_*.png')
    for png_path in sorted(glob.glob(pattern)):
        display(Image(filename=png_path))

## Alternative k results

This section loads post-hoc forced-k final results, when available, from `alternative_k*` folders inside each dimensionality-reduction run. These summaries use only `final/final_metrics.pkl`; the synthetic fold folders are provenance inputs for the merge and are not treated as real fold-level validation results.


In [ ]:

alternative_k_results = {}
alternative_k_summary_rows = []
alternative_k_svm_rows = []


def _cluster_size_summary(labels):
    labels = np.asarray(labels)
    if labels.size == 0:
        return {}
    return pd.Series(labels).value_counts().sort_index().to_dict()


def _load_alternative_k_results():
    loaded = {}
    seen_dirs = set()
    for run_name, run_payload in sorted(dimred_runs.items()):
        search_roots = [Path(run_payload['results_dir']), Path(run_payload['base'])]
        for search_root in search_roots:
            if not search_root.exists():
                continue
            for alt_dir in sorted(search_root.glob('alternative_k*')):
                if not alt_dir.is_dir() or alt_dir.resolve() in seen_dirs:
                    continue
                seen_dirs.add(alt_dir.resolve())
                final_path = alt_dir / 'final' / 'final_metrics.pkl'
                failure_path = alt_dir / 'merge_failure.json'
                result_key = f'{run_name}:{alt_dir.name}'
                payload = {
                    'dimred_option': run_name,
                    'alternative_k': alt_dir.name,
                    'result_key': result_key,
                    'alt_dir': str(alt_dir),
                    'final_metrics_path': str(final_path),
                    'final_metrics': None,
                    'merge_failure_path': str(failure_path) if failure_path.exists() else None,
                    'provenance_files': {
                        name: str(alt_dir / name)
                        for name in [
                            'synthetic_fold_manifest.csv',
                            'candidate_ranking_k.csv',
                            'selected_fold_candidates.csv',
                            'selection_summary.json',
                        ]
                        if (alt_dir / name).exists()
                    },
                }
                if final_path.exists():
                    with open(final_path, 'rb') as f:
                        payload['final_metrics'] = pickle.load(f)
                loaded[result_key] = payload
    return loaded


alternative_k_results = _load_alternative_k_results()

if not alternative_k_results:
    print('No alternative_k* final result folders were found under the loaded dimensionality-reduction runs.')
else:
    for result_key, payload in alternative_k_results.items():
        run_metrics = payload['final_metrics']
        if run_metrics is None:
            alternative_k_summary_rows.append({
                'result_key': result_key,
                'dimred_option': payload['dimred_option'],
                'alternative_k': payload['alternative_k'],
                'final_metrics_found': False,
                'merge_failure_path': payload['merge_failure_path'],
            })
            continue

        diag = run_metrics.get('final_stability_SUM_MAT_full', {})
        labels = np.asarray(run_metrics.get('final_labels', []))
        cluster_sizes = run_metrics.get('final_cluster_sizes') or run_metrics.get('cluster_sizes') or _cluster_size_summary(labels)
        alternative_k_summary_rows.append({
            'result_key': result_key,
            'dimred_option': payload['dimred_option'],
            'alternative_k': payload['alternative_k'],
            'final_metrics_found': True,
            'n_subjects': int(labels.size),
            'n_clusters': int(pd.Series(labels).nunique()) if labels.size else np.nan,
            'cluster_sizes': cluster_sizes,
            'final_params': run_metrics.get('final_params'),
            'final_quality': _safe_numeric(run_metrics.get('final_quality')),
            'final_stability': _safe_numeric(run_metrics.get('final_stability')),
            'final_stability_ari': _safe_numeric(run_metrics.get('final_stability_ari', run_metrics.get('final_stability'))),
            'final_ccc': _safe_numeric(diag.get('CCC', run_metrics.get('final_ccc'))),
            'final_pac': _safe_numeric(diag.get('PAC', run_metrics.get('final_pac'))),
            'final_quality_pvalue': _safe_numeric(run_metrics.get('final_quality_pvalue')),
            'final_stability_ari_pvalue': _safe_numeric(run_metrics.get('final_stability_ari_pvalue')),
            'final_metrics_path': payload['final_metrics_path'],
        })

        svm_results = run_metrics.get('svm_results', {})
        mean_metrics = svm_results.get('mean_metrics', {}) if isinstance(svm_results, dict) else {}
        if mean_metrics:
            row = {
                'result_key': result_key,
                'dimred_option': payload['dimred_option'],
                'alternative_k': payload['alternative_k'],
            }
            row.update(mean_metrics)
            alternative_k_svm_rows.append(row)

    alternative_k_overview_df = pd.DataFrame(alternative_k_summary_rows)
    print('Alternative forced-k final result overview:')
    display(alternative_k_overview_df)

    if alternative_k_svm_rows:
        alternative_k_svm_df = pd.DataFrame(alternative_k_svm_rows)
        print('Alternative forced-k SVM mean metrics:')
        display(alternative_k_svm_df)

    for result_key, payload in alternative_k_results.items():
        print(f'\n=== Alternative k final result: {result_key} ===')
        if payload['merge_failure_path']:
            print('Merge failure file present:', payload['merge_failure_path'])
            try:
                with open(payload['merge_failure_path']) as f:
                    display(json.load(f))
            except Exception as err:
                print('Could not read merge failure JSON:', err)

        for provenance_name, provenance_path in payload['provenance_files'].items():
            print(f'{provenance_name}: {provenance_path}')
            try:
                if provenance_name.endswith('.csv'):
                    provenance_df = pd.read_csv(provenance_path)
                    display(provenance_df.head(20))
                elif provenance_name.endswith('.json'):
                    with open(provenance_path) as f:
                        display(json.load(f))
            except Exception as err:
                print(f'Could not display {provenance_name}: {err}')

        run_metrics = payload['final_metrics']
        if run_metrics is None:
            print('No final_metrics.pkl was available, skipping plots.')
            continue

        alt_plot_dir = os.path.join(
            dimred_runs[payload['dimred_option']]['plots_dir'],
            'alternative_k_results',
            payload['alternative_k'],
        )
        os.makedirs(alt_plot_dir, exist_ok=True)

        plot_dimred_consensus_matrix(
            run_name=result_key,
            run_metrics=run_metrics,
            out_dir=alt_plot_dir,
        )

        try:
            cc_preprocessed_alt_k = build_cc_preprocessed_for_dimred_run(run_metrics)
        except Exception as err:
            print(f'{result_key}: could not prepare CC comparison data, skipping difference plots: {err}')
            continue

        feature_out_dir = os.path.join(alt_plot_dir, 'merged_feature_differences_chr_vs_cc')
        os.makedirs(feature_out_dir, exist_ok=True)
        file_prefix = safe_name(f'{result_key}_chr_vs_cc')

        try:
            feature_stats = make_chr_cc_feature_boxplots(
                chr_df=run_metrics['data'],
                clusters=run_metrics['final_labels'],
                cc_df=cc_preprocessed_alt_k,
                top_k=30,
                out_dir=feature_out_dir,
                file_prefix=file_prefix,
                cc_label='CC',
                title_prefix=f'{result_key}: top features by forced-k subgroup',
            )
            feature_stats_path = os.path.join(feature_out_dir, f'{file_prefix}_feature_stats.csv')
            feature_stats.to_csv(feature_stats_path, index=False)
            print('Saved feature stats to:', feature_stats_path)
            _display_saved_boxplot_pages(feature_out_dir, file_prefix)
        except Exception as err:
            print(f'{result_key}: could not create CHR-vs-CC boxplots: {err}')

        try:
            plot_subgroup_feature_profiles_by_modality(
                data_by_modality={f'Simple clustering ({result_key})': run_metrics['data']},
                labels_by_modality={f'Simple clustering ({result_key})': run_metrics['final_labels']},
                control_data_by_modality={f'Simple clustering ({result_key})': cc_preprocessed_alt_k},
                plots_dir=alt_plot_dir,
                subject_id_column='src_subject_id',
                sample_label=f'discovery_{safe_name(result_key)}',
                subgroup_label='Subgroup',
                control_label='Community controls',
                control_color=CC_COLOR,
                plot_kinds=('line', 'dot', 'heatmap'),
                show=True,
            )
        except Exception as err:
            print(f'{result_key}: could not create subgroup profile plots: {err}')


## Alternative dimensionality-reduction results

For every dimensionality-reduction option that was not selected as the primary downstream run, this section shows the final consensus matrix and the same CHR-vs-CC feature-difference plots used above: boxplots plus subgroup profile plots.

In [ ]:

alternative_dimred_results = {}
alternative_run_names = _alternative_run_names()

if not alternative_run_names:
    print('No alternative dimensionality-reduction runs were loaded.')

for run_name in alternative_run_names:
    run_payload = dimred_runs[run_name]
    run_metrics = run_payload['final_metrics']
    run_plots_dir = run_payload['plots_dir']

    print(f'\n=== Alternative dimensionality-reduction option: {run_name} ===')
    plot_dimred_consensus_matrix(
        run_name=run_name,
        run_metrics=run_metrics,
        out_dir=os.path.join(run_plots_dir, 'alternative_results'),
    )

    try:
        cc_preprocessed_alt = build_cc_preprocessed_for_dimred_run(run_metrics)
    except Exception as err:
        print(f'{run_name}: could not prepare CC comparison data, skipping difference plots: {err}')
        continue

    feature_out_dir = os.path.join(
        run_plots_dir,
        'alternative_results',
        'merged_feature_differences_chr_vs_cc',
    )
    os.makedirs(feature_out_dir, exist_ok=True)
    file_prefix = f'{run_name}_chr_vs_cc'

    feature_stats = None
    try:
        feature_stats = make_chr_cc_feature_boxplots(
            chr_df=run_metrics['data'],
            clusters=run_metrics['final_labels'],
            cc_df=cc_preprocessed_alt,
            top_k=30,
            out_dir=feature_out_dir,
            file_prefix=file_prefix,
            cc_label='CC',
            title_prefix=f'{run_name}: top features by simple-cluster subgroup',
        )
        feature_stats_path = os.path.join(feature_out_dir, f'{file_prefix}_feature_stats.csv')
        feature_stats.to_csv(feature_stats_path, index=False)
        print('Saved feature stats to:', feature_stats_path)
        _display_saved_boxplot_pages(feature_out_dir, file_prefix)
    except Exception as err:
        print(f'{run_name}: could not create CHR-vs-CC boxplots: {err}')

    profile_summaries = None
    try:
        profile_summaries = plot_subgroup_feature_profiles_by_modality(
            data_by_modality={f'Simple clustering ({run_name})': run_metrics['data']},
            labels_by_modality={f'Simple clustering ({run_name})': run_metrics['final_labels']},
            control_data_by_modality={f'Simple clustering ({run_name})': cc_preprocessed_alt},
            plots_dir=os.path.join(run_plots_dir, 'alternative_results'),
            subject_id_column='src_subject_id',
            sample_label=f'discovery_{run_name}',
            subgroup_label='Subgroup',
            control_label='Community controls',
            control_color=CC_COLOR,
            plot_kinds=('line', 'dot', 'heatmap'),
            show=True,
        )
    except Exception as err:
        print(f'{run_name}: could not create subgroup profile plots: {err}')

    alternative_dimred_results[run_name] = {
        'cc_preprocessed': cc_preprocessed_alt,
        'feature_stats': feature_stats,
        'profile_summaries': profile_summaries,
    }


## SVM results

### Final labels

#### Accuracy

In [ ]:
for metric in final_metrics['svm_results']['mean_metrics']:
    print(f"SVM Mean {metric}: {final_metrics['svm_results']['mean_metrics'][metric]}")


#### Uncertainty

In [ ]:
final_metrics['svm_results']['oof_uncertainty']


In [ ]:
# Find most confident mistakes 
df = final_metrics['svm_results']['oof_uncertainty'] 

df_bad = df[(df["y_true"] != df["y_pred"]) & (df["confidence"] > 0.9)]
df_bad.sort_values("confidence", ascending=False).head(20)


In [ ]:
# Find most uncertain predictions
df_uncertain = df.sort_values(["confidence", "entropy"], ascending=[True, False])
df_uncertain.head(20)


In [ ]:
correct = df["y_true"] == df["y_pred"]

plt.figure()
plt.hist(df.loc[correct, "confidence"].dropna(), bins=30, alpha=0.7, label="correct")
plt.hist(df.loc[~correct, "confidence"].dropna(), bins=30, alpha=0.7, label="wrong")
plt.xlabel("Confidence (max predicted probability)")
plt.ylabel("Count")
plt.title("Confidence distribution: correct vs wrong")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
df2 = df.dropna(subset=["confidence"]).copy()
bins = np.linspace(0, 1, 11)

df2["bin"] = pd.cut(df2["confidence"], bins=bins, include_lowest=True)
grp = df2.groupby("bin", observed=True)

acc = grp.apply(lambda g: (g["y_true"] == g["y_pred"]).mean())
cnt = grp.size()

# Midpoints only for bins that exist
x = np.array([interval.mid for interval in acc.index])

plt.figure()
plt.plot(x, acc.values, marker="o")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Confidence bin midpoint")
plt.ylabel("Observed accuracy")
plt.title("Accuracy vs confidence (OOF)")
plt.tight_layout()
plt.show()

print(pd.DataFrame({"bin": acc.index.astype(str), "bin_mid": x, "accuracy": acc.values, "n": cnt.values}))


In [ ]:
df2 = df.dropna(subset=["confidence"]).copy()
df2["correct"] = (df2["y_true"] == df2["y_pred"]).astype(int)

thresholds = np.linspace(0, 1, 101)
coverage = []
error_rate = []

for t in thresholds:
    kept = df2[df2["confidence"] >= t]
    if len(kept) == 0:
        coverage.append(0.0)
        error_rate.append(np.nan)
        continue
    coverage.append(len(kept) / len(df2))
    error_rate.append(1 - kept["correct"].mean())

plt.figure()
plt.plot(coverage, error_rate)
plt.xlabel("Coverage (fraction kept)")
plt.ylabel("Error rate among kept samples")
plt.title("Reject option: trade coverage for accuracy")
plt.tight_layout()
plt.show()


#### Variable contribution

In [ ]:

meta_svm = final_metrics['svm_results']['feature_importance_meta']
print(f"Model information: {meta_svm}")

feat_imp_mean = final_metrics['svm_results']['feature_importance_mean']
feat_imp_std  = final_metrics['svm_results']['feature_importance_std']

# If loaded from packed results (dict), convert to Series
if isinstance(feat_imp_mean, dict):
    feat_imp_mean = pd.Series(feat_imp_mean, dtype=float)
if isinstance(feat_imp_std, dict):
    feat_imp_std = pd.Series(feat_imp_std, dtype=float)

# Build tidy table
df_imp = (
    pd.DataFrame({
        "feature": feat_imp_mean.index,
        "importance_mean": feat_imp_mean.values,
        "importance_std": feat_imp_std.reindex(feat_imp_mean.index).values if feat_imp_std is not None else None,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

df_imp.head(20)


In [ ]:
import matplotlib.pyplot as plt
from Utils import display_feature_name, order_simpleclust_features

top_n = 47
selected_features = order_simpleclust_features(df_imp.head(top_n)["feature"])
plot_df = (
    df_imp.set_index("feature").loc[selected_features].reset_index().iloc[::-1]
)  # reverse so the requested order reads from top to bottom

plt.figure()
plt.barh(plot_df["feature"].map(display_feature_name), plot_df["importance_mean"],
         xerr=plot_df["importance_std"] if "importance_std" in plot_df else None)
plt.xlabel("Feature contribution (mean)")
plt.ylabel("Feature")
plt.title(f"Top {top_n} feature contributions ({meta_svm.get('method','')}, kernel={meta_svm.get('kernel','')})")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure()
plt.hist(df_imp["importance_mean"].values, bins=50)
plt.xlabel("Feature contribution (mean)")
plt.ylabel("Count")
plt.title("Distribution of feature contributions")
plt.tight_layout()
plt.show()


# Apply to test set

## Prepare data

In [ ]:
# Only keep relevant modalities
modalities_keep = ["Psychoticism", "Detachment", "Functioning", "Internalising", "Cognition"]

vars_to_keep = meta["ElementName"][meta["Modality"].isin(modalities_keep)].tolist()
# Filter the data if the columns are present in cleaned_discovery. Also include src_subject_id
vars_to_keep.append("src_subject_id")
vars_to_keep.append("phenotype")
cleaned_test = cleaned_test[cleaned_test.columns.intersection(vars_to_keep)]


## Apply SVM models

Get the test data

In [ ]:
Test_df = pd.read_csv(
    "path/to/simpleclust_data/cleaned_test_data_simpleclust.csv"
)


Apply preprocessing

In [ ]:
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import apply_dimensionality_reduction_to_new_data, apply_preprocessing_to_new_data

subject_id_column = 'src_subject_id'

preproc_artifact = final_metrics.get('preprocessing_details')
if preproc_artifact is None:
    raise ValueError('Selected final_metrics does not contain preprocessing_details; cannot align test data to the trained SVM feature space.')

# Reuse the fitted discovery preprocessing artifact so the independent test
# sample has the same columns, encodings, imputation, and scaling as training.
ae_data, subject_id_list, dict_final = apply_preprocessing_to_new_data(
    Test_df.copy(),
    meta,
    preproc_artifact,
    subject_id_column=subject_id_column,
)

modalities_for_svm = list(preproc_artifact.get('modalities_in_output', dict_final.keys()))
modalities_for_svm = [mod for mod in modalities_for_svm if mod in dict_final]
if not modalities_for_svm:
    raise ValueError('No preprocessed test modalities are available for SVM prediction.')

subject_ids_for_svm = dict_final[modalities_for_svm[0]][subject_id_column].tolist()
for modality in modalities_for_svm[1:]:
    modality_ids = dict_final[modality][subject_id_column].tolist()
    if modality_ids != subject_ids_for_svm:
        raise ValueError(f'Subject-ID order mismatch between {modalities_for_svm[0]} and {modality} after preprocessing.')

def _merge_validation_modalities_for_svm(feature_dict, modalities_to_use, subject_ids, subject_id_column='src_subject_id'):
    merged = None
    for modality in modalities_to_use:
        if modality not in feature_dict:
            continue
        modality_df = feature_dict[modality].copy()
        if subject_id_column not in modality_df.columns:
            raise ValueError(f"{modality} validation data is missing {subject_id_column}.")
        if modality_df[subject_id_column].tolist() != subject_ids:
            modality_df = (
                modality_df
                .set_index(subject_id_column)
                .loc[subject_ids]
                .reset_index()
            )
        feature_columns = [column for column in modality_df.columns if column != subject_id_column]
        modality_df = modality_df.rename(columns={
            column: f'{modality}__{column}'
            for column in feature_columns
            if not str(column).startswith(f'{modality}__')
        })
        merged = modality_df if merged is None else merged.merge(
            modality_df,
            on=subject_id_column,
            how='inner',
            validate='one_to_one',
        )
    if merged is None or merged.empty:
        raise ValueError('No preprocessed validation feature data available for SVM prediction.')
    return merged

# Project through the dimensionality-reduction representation used by the
# selected discovery run. This representation is used for latent/PCA/t-SNE
# validation plots. The final SVM may still have been trained on the raw
# preprocessed matrix, so choose the prediction matrix below from the saved
# svm_feature_names contract.
ae_res, X_test_dimred, X_test_by_modality_dimred = apply_dimensionality_reduction_to_new_data(
    dict_final_new=dict_final,
    final_metrics=final_metrics,
    modalities=modalities_for_svm,
    subject_id_column=subject_id_column,
)

X_test_preprocessed = _merge_validation_modalities_for_svm(
    dict_final,
    modalities_for_svm,
    subject_ids_for_svm,
    subject_id_column=subject_id_column,
)
X_test_preprocessed_features = X_test_preprocessed.drop(columns=[subject_id_column], errors='ignore')

svm_feat = final_metrics.get('svm_feature_names')
X_test_by_modality = X_test_by_modality_dimred
if svm_feat is not None and isinstance(X_test_dimred, pd.DataFrame):
    missing_dimred = [c for c in svm_feat if c not in X_test_dimred.columns]
    missing_preprocessed = [c for c in svm_feat if c not in X_test_preprocessed_features.columns]
    if not missing_dimred:
        X_test = X_test_dimred
        svm_prediction_feature_source = 'dimensionality-reduced validation representation'
    elif not missing_preprocessed:
        X_test = X_test_preprocessed_features
        svm_prediction_feature_source = 'preprocessed validation features'
    else:
        X_test = X_test_dimred
        svm_prediction_feature_source = 'dimensionality-reduced validation representation with missing trained features'
        print('Missing trained features in dim-reduced matrix:', missing_dimred[:10])
        print('Missing trained features in preprocessed matrix:', missing_preprocessed[:10])
else:
    X_test = X_test_dimred
    svm_prediction_feature_source = 'dimensionality-reduced validation representation'

_context = final_metrics.get('final_reporting', {}).get('compute_context', {}) if isinstance(final_metrics, dict) else {}
_dimred_by_modality = _context.get('dim_reduction_by_modality', {}) if isinstance(_context, dict) else {}
_dimred_default = _context.get('dim_reduction', 'none') if isinstance(_context, dict) else 'none'
_validation_dimred_methods = {
    mod: str(_dimred_by_modality.get(mod, _dimred_default)).strip().lower()
    for mod in modalities_for_svm
}
print('Validation dimensionality reduction methods:', _validation_dimred_methods)
print('Validation dimensionality reduction complete:', {k: v['final_latent'].shape for k, v in ae_res.items()})
print('SVM prediction feature source:', svm_prediction_feature_source)
print('Integrated SVM validation matrix:', X_test.shape)

svm_feat = final_metrics.get('svm_feature_names')
if svm_feat is not None and isinstance(X_test, pd.DataFrame):
    missing = [c for c in svm_feat if c not in X_test.columns]
    extra = [c for c in X_test.columns if c not in svm_feat]
    print(f'Integrated SVM feature check: expected={len(svm_feat)}, got={X_test.shape[1]}, missing={len(missing)}, extra={len(extra)}')
    if missing:
        print('Missing integrated feature examples:', missing[:10])
    if extra:
        print('Extra integrated feature examples:', extra[:10])





Run model

In [ ]:
svm_final = final_metrics['svm_final_model']


Integrated clusters

In [ ]:
import pandas as pd
import numpy as np

svm_final = final_metrics['svm_final_model']

# Align to the exact feature order used when the final SVM was trained. Missing
# features indicate a preprocessing/projection mismatch and should not be hidden
# by zero-filled columns.
feat = final_metrics.get('svm_feature_names', None)
if feat is not None and isinstance(X_test, pd.DataFrame):
    missing = [c for c in feat if c not in X_test.columns]
    extra = [c for c in X_test.columns if c not in feat]
    if missing:
        raise ValueError(f'Test SVM matrix is missing {len(missing)} trained features; examples: {missing[:10]}')
    if extra:
        print(f'Dropping {len(extra)} test features not used by the trained SVM; examples: {extra[:10]}')
    X_test_aligned = X_test.loc[:, feat]
else:
    X_test_aligned = X_test

y_pred, proba, confidence, entropy, margin = svm_predict_with_uncertainty(svm_final, X_test_aligned)

pred_final = pd.DataFrame({
    'src_subject_id': subject_ids_for_svm,
    'y_pred': y_pred,
    'confidence': confidence,
    'entropy': entropy,
    'margin': margin,
})

# Optional: add per-class probabilities for debugging.
if proba is not None:
    proba_df = pd.DataFrame(proba, columns=[f'p_{c}' for c in svm_final.classes_])
    pred_final = pd.concat([pred_final, proba_df], axis=1)

print('Integrated SVM predicted classes:', pd.Series(y_pred).astype(str).value_counts().sort_index().to_dict())
display(pred_final.head(10))

labels_test_final = pred_final['y_pred'].to_numpy()




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# pred_final: DataFrame with columns
# ['y_pred', 'confidence', 'entropy', 'margin', 'p_0', 'p_1']

df = pred_final.copy()

# --- (Optional) sanity / recompute checks ---
# If you want to ensure these are consistent with p_0/p_1, uncomment:
# df["confidence_chk"] = np.maximum(df["p_0"], df["p_1"])
# df["margin_chk"] = np.abs(df["p_1"] - df["p_0"])
# eps = 1e-12
# df["entropy_chk"] = -(df["p_0"] * np.log(df["p_0"] + eps) + df["p_1"] * np.log(df["p_1"] + eps))

# --- 1) Histograms ---
plt.figure()
plt.hist(df["p_1"].astype(float), bins=30)
plt.xlabel("Predicted probability p(class=1)")
plt.ylabel("Count")
plt.title("p_1 distribution (unlabeled hold-out)")
plt.show()

plt.figure()
plt.hist(df["confidence"].astype(float), bins=30)
plt.xlabel("Confidence = max(p_0, p_1)")
plt.ylabel("Count")
plt.title("Confidence distribution (unlabeled hold-out)")
plt.show()

plt.figure()
plt.hist(df["entropy"].astype(float), bins=30)
plt.xlabel("Entropy (higher = more uncertain)")
plt.ylabel("Count")
plt.title("Entropy distribution (unlabeled hold-out)")
plt.show()

plt.figure()
plt.hist(df["margin"].astype(float), bins=30)
plt.xlabel("Margin |p_1 - p_0| (lower = more uncertain)")
plt.ylabel("Count")
plt.title("Margin distribution (unlabeled hold-out)")
plt.show()

# --- 2) Scatter plots ---
plt.figure()
plt.scatter(df["confidence"].astype(float), df["entropy"].astype(float), s=10)
plt.xlabel("Confidence")
plt.ylabel("Entropy")
plt.title("Confidence vs entropy")
plt.show()

plt.figure()
plt.scatter(df["confidence"].astype(float), df["margin"].astype(float), s=10)
plt.xlabel("Confidence")
plt.ylabel("Margin")
plt.title("Confidence vs margin")
plt.show()

# --- 3) Quick numeric summaries (handy for write-up) ---
conf = df["confidence"].astype(float).to_numpy()
print("\n=== Summary (unlabeled) ===")
print(f"N = {len(df)}")
print(f"Confidence mean={conf.mean():.3f}, median={np.median(conf):.3f}, min={conf.min():.3f}, max={conf.max():.3f}")
for thr in [0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    print(f"Fraction with confidence ≥ {thr}: {(conf >= thr).mean():.3f}")

# --- 4) Show most-uncertain rows for manual inspection ---
# (lowest confidence, highest entropy, lowest margin)
print("\nLowest confidence (top 10):")
display(df.sort_values("confidence", ascending=True).head(10))

print("\nHighest entropy (top 10):")
display(df.sort_values("entropy", ascending=False).head(10))

print("\nLowest margin (top 10):")
display(df.sort_values("margin", ascending=True).head(10))


## Visualise test labels

### Differences in original variables - final labels

In [ ]:
from sklearn.feature_selection import f_classif
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from Utils import display_feature_name, order_simpleclust_features

# Create directory for differences in original/preprocessed feature plots.
feature_diff_dir = os.path.join(plots_dir, 'merged_feature_differences')
os.makedirs(feature_diff_dir, exist_ok=True)

def _merge_validation_feature_dict(feature_dict, modalities_to_use, subject_ids, subject_id_column='src_subject_id'):
    merged = None
    for modality in modalities_to_use:
        if modality not in feature_dict:
            continue
        modality_df = feature_dict[modality].copy()
        if subject_id_column not in modality_df.columns:
            raise ValueError(f"{modality} validation data is missing {subject_id_column}.")
        if modality_df[subject_id_column].tolist() != subject_ids:
            modality_df = (
                modality_df
                .set_index(subject_id_column)
                .loc[subject_ids]
                .reset_index()
            )
        feature_columns = [column for column in modality_df.columns if column != subject_id_column]
        modality_df = modality_df.rename(columns={
            column: f'{modality}__{column}'
            for column in feature_columns
            if not str(column).startswith(f'{modality}__')
        })
        merged = modality_df if merged is None else merged.merge(
            modality_df,
            on=subject_id_column,
            how='inner',
            validate='one_to_one',
        )
    if merged is None or merged.empty:
        raise ValueError('No validation feature data available for ANOVA plots.')
    return merged

modalities_for_feature_plots = (
    modalities_for_svm
    if 'modalities_for_svm' in globals() and modalities_for_svm
    else list(dict_final.keys())
)
subject_ids_for_feature_plots = (
    subject_ids_for_svm
    if 'subject_ids_for_svm' in globals()
    else dict_final[modalities_for_feature_plots[0]][subject_id_column].tolist()
)

# dict_final is a modality dictionary. Merge it before using DataFrame methods.
df = _merge_validation_feature_dict(
    dict_final,
    modalities_for_feature_plots,
    subject_ids_for_feature_plots,
    subject_id_column=subject_id_column,
)
clusters = np.asarray(labels_test_final)
if len(clusters) != len(df):
    raise ValueError(f'Cluster-label length ({len(clusters)}) does not match validation feature rows ({len(df)}).')
if pd.Series(clusters).nunique(dropna=True) < 2:
    raise ValueError('ANOVA feature plots require at least two predicted SVM classes.')

feature_df = df.drop(columns=[subject_id_column], errors='ignore').select_dtypes(include=[np.number])
feature_df = feature_df.loc[:, feature_df.nunique(dropna=False) > 1]
if feature_df.empty:
    raise ValueError('No non-constant numeric validation features are available for ANOVA plots.')

# --- Feature Ranking (ANOVA F-test) ---
f_vals, _ = f_classif(feature_df, clusters)
f_df = (
    pd.DataFrame({'feature': feature_df.columns, 'f_value': f_vals})
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['f_value'])
    .sort_values('f_value', ascending=False)
)
if f_df.empty:
    raise ValueError('ANOVA did not produce any finite feature scores.')

# --- Barplot of Top Features ---
top_k = min(50, len(f_df))
plt.figure(figsize=(10, 6))
selected_features = order_simpleclust_features(f_df.head(top_k)['feature'])
f_plot_df = f_df.set_index('feature').loc[selected_features].reset_index().assign(
    feature_label=lambda frame: frame['feature'].map(display_feature_name)
)
sns.barplot(
    x='f_value',
    y='feature_label',
    data=f_plot_df,
    palette=sns.color_palette(n_colors=max(top_k, 1)),
)
plt.title(f'Top {top_k} Discriminative Features (ANOVA F-value)')
plt.xlabel('F-value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# --- Boxplots + Scatter Points of Top Features ---
top_features = selected_features
max_cols = 3
n_feats = len(top_features)
n_rows = int(np.ceil(n_feats / max_cols))
fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5), squeeze=False)
axes = axes.flatten()
cluster_palette = sns.color_palette(n_colors=pd.Series(clusters).nunique())

for i, feat in enumerate(top_features):
    ax = axes[i]
    sns.boxplot(x=clusters, y=feature_df[feat], palette=cluster_palette, ax=ax)
    sns.stripplot(x=clusters, y=feature_df[feat], color='black', size=4, jitter=True, alpha=0.5, ax=ax)
    ax.set_title(display_feature_name(feat), fontsize=14)
    ax.set_xlabel('', fontsize=22)
    ax.set_ylabel('', fontsize=22)
    ax.tick_params(axis='both', labelsize=14)

for j in range(n_feats, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Distribution of Top Features by Predicted SVM Class\n(with individual data points)',
    y=1.02,
    fontsize=18,
)
plt.tight_layout()
# fig.savefig(os.path.join(feature_diff_dir, 'top_features.png'))
plt.show()

# --- Validation plots using discovery-trained SVM labels ---
# These figures intentionally reuse labels_test_final, i.e. the classes assigned
# by the discovery-trained final SVM, and the row-aligned validation feature
# table built above.
import glob
import Utils
from IPython.display import Image, display
from Utils import apply_preprocessing_to_new_data, make_chr_cc_feature_boxplots, plot_latent_embeddings, plot_subgroup_feature_profiles_by_modality

validation_plot_dir = os.path.join(plots_dir, 'validation_test_label_plots')
validation_cluster_feature_dir = os.path.join(validation_plot_dir, 'cluster_variable_differences')
validation_extra_feature_dir = os.path.join(validation_plot_dir, 'extra_variable_differences')
validation_latent_dir = os.path.join(validation_plot_dir, 'latent_pca_tsne')
validation_matrix_dir = os.path.join(validation_plot_dir, 'matrix_plots')
for _plot_dir in [
    validation_plot_dir,
    validation_cluster_feature_dir,
    validation_extra_feature_dir,
    validation_latent_dir,
    validation_matrix_dir,
]:
    os.makedirs(_plot_dir, exist_ok=True)

validation_labels_for_plots = np.asarray(labels_test_final)
validation_subject_ids_for_plots = df[subject_id_column].astype(str).to_numpy()
validation_data_for_plots = df.copy()
validation_feature_df = feature_df.copy()

validation_label_counts = pd.Series(validation_labels_for_plots).astype(str).value_counts().sort_index()
print('Validation plot labels from discovery-trained SVM:', validation_label_counts.to_dict())

# Prepare CC controls for validation plots with the same fitted preprocessing
# artifact used above for the validation CHR sample. Prefer the held-out/test CC
# split when it is available; fall back to discovery CC only when no test CC rows
# are present in the notebook state.
def _validation_cc_source_dataframe():
    if 'cleaned_test_CC' in globals() and isinstance(cleaned_test_CC, pd.DataFrame) and not cleaned_test_CC.empty:
        return cleaned_test_CC.copy(), 'held-out/test CC split'
    if 'test_data_CC' in globals() and isinstance(test_data_CC, pd.DataFrame) and not test_data_CC.empty:
        return test_data_CC.copy(), 'held-out/test CC split before missingness filtering'
    if 'cleaned_discovery_CC' in globals() and isinstance(cleaned_discovery_CC, pd.DataFrame) and not cleaned_discovery_CC.empty:
        return cleaned_discovery_CC.copy(), 'discovery CC fallback'
    return None, None


def _preprocess_cc_like_validation_chr(cc_source_df, cc_source_label):
    if cc_source_df is None or cc_source_df.empty:
        print('No CC dataframe is available for validation plots; CC overlays will be skipped.')
        return None
    if 'preproc_artifact' not in globals() or preproc_artifact is None:
        raise ValueError('preproc_artifact is required to align validation CC to the validation CHR feature space.')
    _, _, dict_final_validation_cc = apply_preprocessing_to_new_data(
        cc_source_df.copy(),
        meta,
        preproc_artifact,
        subject_id_column=subject_id_column,
    )
    validation_cc = _merge_validation_feature_dict(
        dict_final_validation_cc,
        modalities_for_feature_plots,
        dict_final_validation_cc[modalities_for_feature_plots[0]][subject_id_column].tolist(),
        subject_id_column=subject_id_column,
    )
    missing_validation_cc_features = [
        column for column in validation_data_for_plots.columns
        if column != subject_id_column and column not in validation_cc.columns
    ]
    if missing_validation_cc_features:
        raise ValueError(
            f'Validation CC preprocessing is missing {len(missing_validation_cc_features)} validation CHR features; '
            f'examples: {missing_validation_cc_features[:10]}'
        )
    validation_cc = validation_cc[[subject_id_column] + [c for c in validation_data_for_plots.columns if c != subject_id_column]]
    print(f'Validation CC preprocessing complete using {cc_source_label}:', validation_cc.shape)
    return validation_cc

validation_cc_source_df, validation_cc_source_label = _validation_cc_source_dataframe()
validation_cc_preprocessed = _preprocess_cc_like_validation_chr(
    validation_cc_source_df,
    validation_cc_source_label,
)
if validation_cc_source_label == 'discovery CC fallback':
    print('Warning: no held-out/test CC rows were available; validation plots use discovery CC as the control reference.')

# 1) Validation CHR-vs-CC boxplots and feature stats, matching the discovery
# make_chr_cc_feature_boxplots path.
validation_feature_stats = None
if validation_cc_preprocessed is not None:
    validation_feature_stats = make_chr_cc_feature_boxplots(
        chr_df=validation_data_for_plots,
        clusters=validation_labels_for_plots,
        cc_df=validation_cc_preprocessed,
        top_k=30,
        out_dir=validation_cluster_feature_dir,
        file_prefix='validation_chr_vs_cc',
        cc_label='CC',
        title_prefix='Validation top features by discovery-trained SVM subgroup',
    )
    validation_feature_stats_path = os.path.join(validation_cluster_feature_dir, 'validation_chr_vs_cc_feature_stats.csv')
    validation_feature_stats.to_csv(validation_feature_stats_path, index=False)
    print('Saved validation CHR-vs-CC feature stats to:', validation_feature_stats_path)
    for png_path in sorted(glob.glob(os.path.join(validation_cluster_feature_dir, 'validation_chr_vs_cc_boxplots_page_*.png'))):
        display(Image(filename=png_path))

    if 'make_original_feature_pairwise_tests' in globals() and 'plot_original_feature_pairwise_tiles' in globals():
        validation_cluster_pairwise_stats = make_original_feature_pairwise_tests(
            chr_df=validation_data_for_plots,
            clusters=validation_labels_for_plots,
            ranked_stats=validation_feature_stats,
            subject_id_column=subject_id_column,
        )
        validation_cluster_pairwise_path = os.path.join(
            validation_cluster_feature_dir,
            'validation_chr_vs_cc_all_pairwise_feature_tests.csv',
        )
        validation_cluster_pairwise_stats.to_csv(validation_cluster_pairwise_path, index=False)
        print('Saved validation clustering-variable pairwise post-hoc tests to:', validation_cluster_pairwise_path)
        display(validation_cluster_pairwise_stats.head(30))
        plot_original_feature_pairwise_tiles(
            pairwise_df=validation_cluster_pairwise_stats,
            ranked_stats=validation_feature_stats,
            out_dir=validation_cluster_feature_dir,
            file_prefix='validation_chr_vs_cc',
            top_n=30,
        )
    else:
        print(
            'Skipping clustering-variable pairwise heatmap because '
            'make_original_feature_pairwise_tests/plot_original_feature_pairwise_tiles are not loaded. '
            'Run the discovery pairwise-test helper cell first.'
        )
else:
    print('Skipping validation CHR-vs-CC boxplots because no aligned CC controls are available.')

# 2) Profile plots for validation labels across the same preprocessed cluster
# variables, with CC included using the same control overlay as discovery.
try:
    validation_profile_summaries = plot_subgroup_feature_profiles_by_modality(
        data_by_modality={'Validation simple clustering': validation_data_for_plots},
        labels_by_modality={'Validation simple clustering': validation_labels_for_plots},
        control_data_by_modality=(
            {'Validation simple clustering': validation_cc_preprocessed}
            if validation_cc_preprocessed is not None else None
        ),
        plots_dir=validation_plot_dir,
        subject_id_column=subject_id_column,
        sample_label='validation',
        subgroup_label='Subgroup',
        control_label='Community controls',
        control_color=CC_COLOR,
        plot_kinds=('line', 'dot', 'heatmap'),
        show=True,
    )
except Exception as err:
    print('Could not create validation profile plots:', err)

# 3) Heatmap of validation cluster-variable subgroup differences. This mirrors
# the profile heatmap numerically and includes a CC row when controls are
# available, using the same fitted preprocessing scale.
def _plot_validation_feature_mean_heatmap(feature_matrix, labels, out_dir, file_prefix, title, features=None, control_matrix=None):
    if features is None:
        features = order_simpleclust_features(feature_matrix.columns)
    features = [feature for feature in features if feature in feature_matrix.columns]
    if not features:
        print(f'{title}: no available features for heatmap.')
        return None
    plot_matrix = feature_matrix[features].apply(pd.to_numeric, errors='coerce')
    plot_matrix = plot_matrix.loc[:, plot_matrix.notna().any(axis=0)]
    if plot_matrix.empty:
        print(f'{title}: no numeric non-empty features for heatmap.')
        return None
    use_features = plot_matrix.columns.tolist()
    combined_for_scale = plot_matrix.copy()
    control_numeric = None
    if control_matrix is not None:
        control_numeric = (
            control_matrix.reindex(columns=use_features)
            .apply(pd.to_numeric, errors='coerce')
            .dropna(axis=0, how='all')
        )
        if not control_numeric.empty:
            combined_for_scale = pd.concat([combined_for_scale, control_numeric], ignore_index=True)
    standardized = plot_matrix.sub(combined_for_scale.mean(axis=0), axis=1).div(
        combined_for_scale.std(axis=0, ddof=0).replace(0, np.nan),
        axis=1,
    ).fillna(0)
    heatmap_df = standardized.assign(_subgroup=pd.Series(labels).astype(str).to_numpy())
    mean_matrix = heatmap_df.groupby('_subgroup', observed=False)[standardized.columns].mean()
    mean_matrix = mean_matrix.reindex(sorted(mean_matrix.index, key=lambda value: str(value)))
    if control_numeric is not None and not control_numeric.empty:
        control_standardized = control_numeric.sub(combined_for_scale.mean(axis=0), axis=1).div(
            combined_for_scale.std(axis=0, ddof=0).replace(0, np.nan),
            axis=1,
        ).fillna(0)
        mean_matrix.loc['CC'] = control_standardized.mean(axis=0)
    mean_matrix.columns = [display_feature_name(feature) for feature in mean_matrix.columns]

    fig_width = min(max(12, 0.38 * mean_matrix.shape[1] + 3), 44)
    fig_height = max(3.5, 0.55 * mean_matrix.shape[0] + 2.2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    sns.heatmap(
        mean_matrix,
        cmap='vlag',
        center=0,
        linewidths=0.25,
        linecolor='white',
        cbar_kws={'label': 'Mean z-score'},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Feature')
    ax.set_ylabel('Predicted validation subgroup / CC')
    ax.tick_params(axis='x', rotation=70, labelsize=8)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        output_path = os.path.join(out_dir, f'{file_prefix}.{ext}')
        fig.savefig(output_path, dpi=300, bbox_inches='tight')
        print('Saved validation feature heatmap to:', output_path)
    plt.show()
    return mean_matrix

validation_cluster_feature_heatmap = _plot_validation_feature_mean_heatmap(
    validation_feature_df,
    validation_labels_for_plots,
    validation_cluster_feature_dir,
    'validation_cluster_variable_subgroup_mean_heatmap',
    'Validation cluster variables by discovery-trained SVM subgroup',
    features=order_simpleclust_features(validation_feature_df.columns),
    control_matrix=(
        validation_cc_preprocessed.drop(columns=[subject_id_column], errors='ignore')
        if validation_cc_preprocessed is not None else None
    ),
)
if validation_cluster_feature_heatmap is not None:
    validation_cluster_feature_heatmap.to_csv(
        os.path.join(validation_cluster_feature_dir, 'validation_cluster_variable_subgroup_mean_heatmap.csv')
    )

# 4) Extra-variable differences for validation labels, using the same
# compare_clusters metadata and plot functions as the discovery sample.
if 'compare_extra_features_by_cluster' in globals():
    try:
        validation_extra_stats, validation_extra_heatmap_df, validation_extra_long_df, validation_extra_pairwise_df, validation_missing_compare_vars = compare_extra_features_by_cluster(
            source_df=Test_df,
            final_data=validation_data_for_plots,
            final_labels=validation_labels_for_plots,
            meta=meta,
            modality='compare_clusters',
            subject_id_column=subject_id_column,
        )
        validation_extra_stats_path = os.path.join(validation_extra_feature_dir, 'validation_compare_clusters_extra_feature_tests.csv')
        validation_extra_pairwise_path = os.path.join(validation_extra_feature_dir, 'validation_compare_clusters_extra_feature_pairwise_tests.csv')
        validation_extra_long_path = os.path.join(validation_extra_feature_dir, 'validation_compare_clusters_extra_feature_subgroup_profiles.csv')
        validation_extra_heatmap_path = os.path.join(validation_extra_feature_dir, 'validation_compare_clusters_extra_feature_heatmap_matrix.csv')
        validation_extra_stats.to_csv(validation_extra_stats_path, index=False)
        validation_extra_pairwise_df.to_csv(validation_extra_pairwise_path, index=False)
        validation_extra_long_df.to_csv(validation_extra_long_path, index=False)
        validation_extra_heatmap_df.to_csv(validation_extra_heatmap_path, index=False)
        print(f'Compared {validation_extra_stats.shape[0]} validation compare_clusters variables across predicted subgroups.')
        if validation_missing_compare_vars:
            print(f'Validation metadata variables not found in Test_df ({len(validation_missing_compare_vars)}): {validation_missing_compare_vars[:10]}')
        print('Saved validation extra-feature test table to:', validation_extra_stats_path)
        print('Saved validation extra-feature pairwise post-hoc table to:', validation_extra_pairwise_path)
        print('Saved validation extra-feature subgroup profile table to:', validation_extra_long_path)
        display(validation_extra_stats.head(30))
        display(validation_extra_pairwise_df.head(30))

        if 'plot_top_feature_profile_dotplot' in globals():
            plot_top_feature_profile_dotplot(
                stats_df=validation_extra_stats,
                long_df=validation_extra_long_df,
                out_dir=validation_extra_feature_dir,
                file_prefix='validation_compare_clusters_extra_features',
                title='Validation compare-cluster variables by predicted subgroup',
                top_n=None,
            )
        if 'plot_pairwise_fdr_tiles' in globals():
            plot_pairwise_fdr_tiles(
                pairwise_df=validation_extra_pairwise_df,
                stats_df=validation_extra_stats,
                out_dir=validation_extra_feature_dir,
                file_prefix='validation_compare_clusters_extra_features',
                title='Validation pairwise post-hoc tests for compare-cluster variables',
                top_n=None,
            )
    except Exception as err:
        print('Could not create validation extra-variable difference plots:', err)
else:
    print('Skipping validation extra-variable plots because compare_extra_features_by_cluster has not been run yet.')

# 5) Latent/PCA/t-SNE plots for validation, using the same Utils helper and
# visual contract as the discovery plots: applied representation, PCA, t-SNE.
validation_latent_metrics = dict(final_metrics)
validation_latent_metrics['ae_res'] = ae_res
validation_latent_metrics['final_labels'] = validation_labels_for_plots
validation_latent_metrics['data'] = validation_data_for_plots
validation_latent_metrics['dim_reduction'] = final_metrics.get(
    'dim_reduction',
    final_metrics.get('dim_reduction_label', best_dimred_option if 'best_dimred_option' in globals() else 'validation'),
)

validation_latent_matrix, validation_latent_source = Utils._extract_final_latent(ae_res) if hasattr(Utils, '_extract_final_latent') else (None, None)
if validation_latent_matrix is not None:
    validation_latent_array = np.asarray(validation_latent_matrix)
    print(
        'Validation latent representation for plotting:',
        {
            'source': validation_latent_source or 'flat',
            'shape': validation_latent_array.shape,
            'min': float(np.nanmin(validation_latent_array)),
            'max': float(np.nanmax(validation_latent_array)),
            'zero_fraction': float(np.mean(np.isclose(validation_latent_array, 0.0))),
        },
    )
    if str(validation_latent_metrics.get('dim_reduction', '')).lower().startswith('sparsepca'):
        expected_sparsepca_components = final_metrics.get('dim_reduction_n_components')
        if expected_sparsepca_components is None:
            expected_sparsepca_components = getattr(final_metrics.get('ae_res', {}).get('spca_model', None), 'n_components', None)
        if expected_sparsepca_components is not None and validation_latent_array.ndim == 2 and validation_latent_array.shape[1] != int(expected_sparsepca_components):
            raise ValueError(
                f'Validation SparsePCA projection has {validation_latent_array.shape[1]} columns, '
                f'but selected SparsePCA run expects {int(expected_sparsepca_components)} components. '
                'Rerun the preprocessing/projection cell after reloading the updated Utils.py.'
            )
        print(
            'SparsePCA validation plot uses the discovery-fitted SparsePCA transform stored in final_metrics/ae_res; '
            'axis-aligned or cross-like structure can occur when SparsePCA components produce many exact or near-zero scores.'
        )
validation_latent_fig = plot_latent_embeddings(
    f'validation_{best_dimred_option}' if 'best_dimred_option' in globals() else 'validation',
    validation_latent_metrics,
    out_dir=validation_latent_dir,
    file_prefix='validation_test_labels',
)
if validation_latent_fig is not None:
    plt.show()

# 6) Matrix plots: do not draw a validation consensus matrix. The discovery
# matrix is a bootstrap co-association/consensus matrix from repeated clustering.
# Here the validation labels are SVM assignments, so no equivalent co-clustering
# matrix exists unless the full resampling/clustering procedure is rerun on the
# validation sample. A raw feature-similarity matrix would be a different object
# and was therefore removed to avoid over-interpreting it.
validation_matrix_note = (
    'No validation consensus matrix was plotted. Discovery matrix plots show bootstrap '
    'co-association from the clustering ensemble. Validation labels here are assigned by '
    'the discovery-trained SVM, so there is no comparable consensus matrix without rerunning '
    'a validation clustering/resampling analysis.'
)
print(validation_matrix_note)
with open(os.path.join(validation_matrix_dir, 'validation_matrix_plot_note.txt'), 'w') as f:
    f.write(validation_matrix_note + '\n')



# Longitudinal analyses across months 1-5

This replaces the old month 2-only comparisons. It fits baseline-cluster mixed models across all available months and maps follow-up observations back onto baseline cluster centroids for discovery and validation samples.


In [ ]:
import os
import importlib
import Utils
Utils = importlib.reload(Utils)
run_longitudinal_multiclust_report = Utils.run_longitudinal_multiclust_report
display_longitudinal_multiclust_results = Utils.display_longitudinal_multiclust_results
load_longitudinal_multiclust_results = Utils.load_longitudinal_multiclust_results

_longitudinal_data_dirs = ['path/to/simpleclust_data']
_subject_id_column = subject_id_column if "subject_id_column" in globals() else "src_subject_id"

_longitudinal_prescient_ids = (
    prescient_ids
    if "prescient_ids" in globals()
    else (set(prescient["src_subject_id"]) if "prescient" in globals() else None)
)

_validation_baseline_data = None
if "dict_final_test" in globals() and isinstance(dict_final_test, dict) and dict_final_test:
    _validation_baseline_data = dict_final_test
elif "dict_final" in globals() and isinstance(dict_final, dict) and dict_final and "labels_test_final" in globals():
    _validation_baseline_data = dict_final

_validation_subject_ids = None
if isinstance(_validation_baseline_data, dict) and _validation_baseline_data:
    _validation_subject_ids = {
        modality: df[_subject_id_column].astype(str).tolist()
        for modality, df in _validation_baseline_data.items()
        if _subject_id_column in df.columns
    }
elif "subject_id_list_test" in globals() and "modalities" in globals():
    _validation_subject_ids = {
        modality: ids
        for modality, ids in zip(modalities, subject_id_list_test)
    }

_validation_domain_labels = None
if "new_test_labels_by_modality" in globals() and new_test_labels_by_modality:
    _validation_domain_labels = new_test_labels_by_modality
elif "labels_test_by_modality" in globals() and labels_test_by_modality:
    _validation_domain_labels = {k: v for k, v in labels_test_by_modality.items() if v is not None}
elif "labels_test_modalities" in globals() and "modalities" in globals():
    _validation_domain_labels = {
        modality: labels
        for modality, labels in zip(modalities, labels_test_modalities)
        if labels is not None
    }

_validation_final_labels = labels_test_final if "labels_test_final" in globals() else None

longitudinal_results = run_longitudinal_multiclust_report(
    final_metrics=final_metrics,
    meta=meta,
    plots_dir=plots_dir,
    data_dirs=_longitudinal_data_dirs,
    prescient_ids=_longitudinal_prescient_ids,
    vars_to_keep=None,
    categorical_columns=cat_vars if "cat_vars" in globals() else None,
    validation_domain_labels=_validation_domain_labels,
    validation_final_labels=_validation_final_labels,
    validation_subject_ids=_validation_subject_ids,
    validation_baseline_data=_validation_baseline_data,
    months=(1, 2, 3, 4, 5),
    subject_id_column=_subject_id_column,
    min_features_per_analysis=2,
    min_features_for_cluster_change=1,
    min_followup_timepoints_per_feature=1,
    min_nonmissing_per_timepoint=8,
    min_group_n=4,
    reuse_existing=False,
)

print("Longitudinal outputs saved under:", longitudinal_results["output_dir"])
print("Month files used:")
for month, path in longitudinal_results["month_files"].items():
    print(f"  month {month}: {path}")
print("Analyses completed:", len(longitudinal_results["analyses"]))
if longitudinal_results["analysis_summary"].empty:
    print("No mixed-model or cluster-change analyses were completed. Check longitudinal_results['preprocessing_report'] for preprocessing failures.")
else:
    display(longitudinal_results["analysis_summary"])





display_longitudinal_multiclust_results(longitudinal_results, max_analyses=20, show_sankey=True)





In [ ]:
_preprocessing_report = longitudinal_results["preprocessing_report"]
_sort_cols = [
    col for col in ["sample", "month", "modality"]
    if col in _preprocessing_report.columns
]
_preprocessing_report_view = (
    _preprocessing_report.sort_values(_sort_cols)
    if _sort_cols else _preprocessing_report
)
display(_preprocessing_report_view.head(30))

if "status" in _preprocessing_report.columns:
    display(
        _preprocessing_report
        .groupby([c for c in ["sample", "status"] if c in _preprocessing_report.columns])
        .size()
        .reset_index(name="n_rows")
    )


In [ ]:
# Display saved longitudinal results without rerunning the analyses.
import importlib
import Utils
importlib.reload(Utils)
from Utils import load_longitudinal_multiclust_results, display_longitudinal_multiclust_results

_existing_longitudinal_results = load_longitudinal_multiclust_results(
    os.path.join(plots_dir, "longitudinal_all_timepoints")
)
display_longitudinal_multiclust_results(
    _existing_longitudinal_results,
    max_analyses=10,
    show_sankey=True,
)
